# ConformalStereo: Cambridge deposits, run 3 (notebook 1.2)

Runs 1 and 2 parsed four deposits of the Goodman group from Apollo, the University of Cambridge repository.
This third run goes back to one of them, "Doubling the power of DP4" (2017), for a quantity the first parser did
not keep: the calculated spin-spin coupling constants. The deposit holds about 3,800 Gaussian outputs with them,
all in folders whose name ends in `_J`.

The run downloads the deposit (a 9.6 GB zip in ten volumes), reads only the files inside the `_J` folders, and
deletes the archives from the scratch disk afterwards. Nothing has to be unpacked by hand.

**Before running**
1. Right panel, *Settings*: switch **Internet** on. Accelerator: None.
2. *Save Version*, then **Save & Run All (Commit)**. The run needs no attention.
   Expect about a quarter of an hour; most of it is download time and depends on the repository.

**After the run**
Open the saved version, go to *Output*, and download the output as one zip (or every file named `cs12_core_NN.zip`).
Send that zip back.

| Key | Deposit | DOI | This run |
| --- | --- | --- | --- |
| `dp4x2` | "Doubling the power of DP4" (2017): 42 compounds | 10.17863/CAM.13222 | coupling constants in the `_J` folders |
| `dp4ai` | DP4-AI: 46 stereo cases at three levels of theory | 10.17863/CAM.47721 | off (complete in runs 1 and 2) |
| `dp5` | DP5, revised: structure reassignment examples | 10.17863/CAM.81332 | off (complete in run 1) |
| `conf2018` | Conformational search and DFT data (Ermanis et al., 2018) | 10.17863/CAM.32463 | off (complete in runs 1 and 2) |

The record pages give CC BY 4.0 as the licence of these deposits; cite the DOIs when the data are used.

What is written for every job step that prints coupling constants: the matrix of total couplings J (Hz) and the
matrix of its Fermi-contact part, each with the atom numbers Gaussian printed for it, next to the route, the SCF
energy and the list of elements. The text of these matrices is kept as printed for the first two outputs of every
folder and for any matrix whose layout is not the expected one, so a reading error can be repaired without another
run. The other job steps of the same folders are written as in run 2, and the small text files next to them (NMR
descriptions with the experimental couplings, PyDP4 outputs) are copied as they are.

Only the cell "1. Settings" is meant to be edited.

In [ ]:
# ---- 1. Settings ---------------------------------------------------------------------------------------------
import os, sys, re, io, json, gzip, time, math, shutil, hashlib, zipfile, tarfile, traceback, collections
import subprocess, multiprocessing

NOTEBOOK_VERSION = '1.2 (2026-11-14)'

# mode: 'parse' = download and parse everything | 'mae' = download, read the Maestro files only
#       'jcoup' = download, read only the members whose path matches J_PATH (the coupling-constant folders)
#       'list' = write the file listing only | 'off' = ignore
# Runs 1 and 2 (versions 1.0 and 1.1, 13 and 14 Nov 2026) parsed all four deposits. This run returns to the 2017
# deposit for the spin-spin coupling constants, which those versions did not keep.
DEPOSITS = [
    dict(key='dp4x2', mode='jcoup', doi='10.17863/CAM.13222', handle='1810/267217', item='b41c395b-8a42-43be-b2f9-ee39522f1133',
         title='Doubling the power of DP4 (2017): 42 compounds, DFT conditions screened'),
    dict(key='dp4ai', mode='off', doi='10.17863/CAM.47721', handle='1810/303161', item='3b9dadf2-4701-49af-af2f-c8491384e199',
         title='DP4-AI: DFT, molecular mechanics and raw NMR data (46 stereo cases, three levels)', skip_names=['NMR_Data.zip']),
    dict(key='dp5', mode='off', doi='10.17863/CAM.81332', handle='1810/334048', item='f4786c07-e546-4e0a-b8fb-c8022997c9de',
         title='DP5, revised: structure reassignment examples'),
    dict(key='conf2018', mode='off', doi='10.17863/CAM.32463', handle='1810/285709', item='53d5cb49-ed72-439d-99b4-7c6e3586e5cb',
         title='Ermanis 2018: conformational search and DFT data at several geometry and energy levels'),
]

ONLY_FILES = []            # e.g. ['AT1.zip', 'AT2.zip'] for a quick trial; empty = every file of the deposit
LOCAL_SOURCES = {}         # manual route: {'dp4ai': ['/kaggle/input/my-upload']} = folders or archives already on disk
MAX_FILE_GB = 14           # a single repository file (or split-zip set) above this size is skipped and reported
MAX_DEPOSIT_GB = 40        # a deposit above this total size is listed, not downloaded
DEADLINE_HOURS = 10.5      # stop starting new archives after this long and package what is done
PART_MB = 20               # size of each result zip
MAE_WINDOW_KJ = 12.0       # conformer geometries are kept for Maestro structures within this energy of the minimum
MAE_MAX_GEOMS = 500        # ... and at most this many per Maestro file
SMALL_FILE_KB = 300        # text files up to this size (structures, NMR descriptions, DP4 outputs) are copied
SMALL_TOTAL_MB = 80        # ... up to this total per deposit, after removing duplicates
BIG_MEMBER_MB = 300        # outputs larger than this are read as first 2 MB + last 64 MB only
HARVEST_MB = 150           # outputs of other programs (NWChem, ORCA, Jaguar): energy and shielding lines are kept, up to this much per deposit
J_PATH = r'_J/'            # mode 'jcoup': only members whose path matches this regular expression are read
J_TERMS = ['total', 'FC']  # coupling matrices that are kept: 'total', 'FC' (Fermi contact), 'SD', 'PSO', 'DSO'
J_RAW_PER_FOLDER = 2       # the printed text of the kept matrices is saved for this many outputs of every folder
J_RAW_MB = 150             # ... and for every matrix with an unexpected layout, up to this much text per deposit

# Test hooks: environment variables override the defaults above. Not needed on Kaggle.
ON_KAGGLE = os.path.isdir('/kaggle/working')
WORK = os.environ.get('CS_WORK', '/kaggle/working' if ON_KAGGLE else os.path.abspath('cs_work'))
API_BASES = [b.strip().rstrip('/') for b in os.environ.get(
    'CS_API_BASES', 'https://api.repository.cam.ac.uk/server/api,https://www.repository.cam.ac.uk/server/api').split(',') if b.strip()]
UI_BASE = os.environ.get('CS_UI_BASE', 'https://www.repository.cam.ac.uk').rstrip('/')
INPUT_DIRS = [d for d in os.environ.get('CS_INPUT_DIRS', '/kaggle/input').split(os.pathsep) if d]
PART_MB = float(os.environ.get('CS_PART_MB', PART_MB))
if os.environ.get('CS_DEPOSITS_JSON'):
    DEPOSITS = json.loads(os.environ['CS_DEPOSITS_JSON'])
if os.environ.get('CS_LOCAL_SOURCES_JSON'):
    LOCAL_SOURCES = json.loads(os.environ['CS_LOCAL_SOURCES_JSON'])
N_WORKERS = int(os.environ.get('CS_WORKERS', max(1, min(4, os.cpu_count() or 1))))
if os.environ.get('CS_OVERRIDES_JSON'):
    globals().update(json.loads(os.environ['CS_OVERRIDES_JSON']))

# Files as listed by the repository in run 1 (13 Nov 2026): name, id, size, MD5. Used for a size check, and as the file
# list if the listing service does not answer.
KNOWN_FILES = json.loads(os.environ['CS_KNOWN_FILES_JSON']) if os.environ.get('CS_KNOWN_FILES_JSON') else {
    'dp4ai': [
        ('NP1.zip', '362208e1-b76e-46df-8c69-e6bd360f3da7', 33486734, '9f2c3df393524a90111417868dc3f707'),
    ],
    'conf2018': [
        ('AldolsGeom.zip', 'd1543564-9eed-48e8-a10a-4a019500223a', 155541916, 'd1364a54ad78e28c29edca427fd28df1'),
        ('ArtemisininGeom.zip', '2c0f3a8f-0d6e-4c84-bda3-d8c782d77ab5', 48598509, '055d6722cf7ef591ae5c01497ca8059a'),
        ('AspergillideGeom.zip', 'ea0f06ac-651a-4990-9177-9edfb2b334b2', 116119184, '539b680150d8f870a9a52b152af15a9c'),
        ('CitrinadinBGeom.zip', 'bc1fbb16-f7be-4a14-8b74-b41f9874e13f', 229471840, '26d82614c2ec04abd659655464c1568e'),
        ('DidanosineGeom.zip', 'c3811a2e-e8c2-42db-a6f7-36734392bc29', 62684014, '465c7c452a70dbedc9c6f8185aa67ae1'),
        ('EntecavirGeom.zip', '17bce622-9cda-46b2-a142-b63a310c1a8f', 112584639, '4ddb2b094d1b22bce5903b280ab20876'),
        ('EzetimibeGeom.zip', '0c14f2c2-a0e4-426e-aba9-15a30a5d949c', 199391643, '9f3ca7f32f5bb14b4bb67d32ccae82f2'),
        ('FormoterolGeom.zip', '5aed3878-ec48-49d8-a3aa-9a93dddcfafe', 104124894, '58104bbab001eb5349cb9e254b8eafc1'),
        ('LopinavirGeom.zip', 'd7749324-5ccd-4e27-aa6b-077fe878ae77', 964669523, 'cd487c71d329dbe3efc1ecfa3aba0bc6'),
        ('OseltamivirGeom.zip', '9c66c5f7-aa4d-480b-bf23-fd6556ddfc63', 1266218334, '81430d0678f70269eb02c134bbdc5866'),
        ('RosuvastatinGeom.zip', '401840a9-7fa5-479e-be71-fbe89bc4b816', 514114744, '222b4a441b0932cdf34aac620ed87bf8'),
        ('SolifenacinGeom.zip', '7aabc043-eb11-4269-8b42-f59f4a1ea5c1', 31445989, '6b4ec71b47e97d3db669d6c3ef4a3fcb'),
        ('TadalafilGeom.zip', '32ff2f32-4e1d-4b5c-bf30-0ccea25aeeec', 22676348, '4e139d0905c675acd19f3d640521e1db'),
        ('TanarifuranonolGeom.zip', '490a82a3-9934-455f-9c32-25e8aaeb9743', 307712982, '56c7d4269d0328604c9d675662ab8740'),
        ('TetrahydrofuransGeom.zip', '37708ba1-a19b-4dd3-b9e3-3bc2e9f69700', 336180384, '0855b041a1d2f6ccd0251a6e01989e8c'),
        ('TiotropiumGeom.zip', 'aa525a85-7f8b-4c65-8bff-06a1f93b795b', 207247877, '934fe8e81115bf5666f899e10c505aad'),
        ('ZidovudineGeom.zip', 'dadcf031-7c89-44ee-b7fa-806dea4e6115', 158980398, '6cda083bb1b5a8cf8040fb4f644defb1'),
        ('DataReadme.txt', 'f8dc6e43-394f-4b43-95f2-57897d492b14', 1910, '46ecb112445485a3f39a8e7c9b92c6f2'),
    ],
    'dp4x2': [
        ('BasisFunc.zip', 'e9a6e024-d8c9-4449-b79b-df04a11f945f', 636285071, 'c3918787d2e340ff8cbfcf733acef52d'),
        ('BasisFunc.z01', 'c9e32212-d5a4-4f29-9ded-c0a49b854c1a', 1073741824, '04f4c5103b5986bec4f0fb7261b7e498'),
        ('BasisFunc.z02', '9c7e04f6-2da3-4e48-8e3f-96bbbb8b80cb', 1073741824, '5c8ed1475cd2b6798853de83a0aff121'),
        ('BasisFunc.z03', '2d3ae495-8e17-4321-9e10-30131e58e2ec', 1073741824, 'e35ea3afe6a7f25215d700f9237af020'),
        ('BasisFunc.z04', 'e63a99ed-d55b-4c85-8b5d-9e839f61a1da', 1073741824, '7844ba87c8ee5791dda543b146260f51'),
        ('BasisFunc.z05', '0b362958-cc69-4358-995a-1da92e7e0dfe', 1073741824, 'a3880b1e750a132fbbcbca3d93d54aab'),
        ('BasisFunc.z06', '4d88106a-9bfe-4444-b251-fecde7b170f8', 1073741824, '370600321746e442d7efadbf07f16f4b'),
        ('BasisFunc.z07', '8ec2387a-ff44-499a-985a-afd31af8c015', 1073741824, '167b98d25c9ff17605ac76ef3272f214'),
        ('BasisFunc.z08', '2ec20eff-4f16-41c5-a7a4-65a60ab0de76', 1073741824, 'c49e4ab4815231e815779b15a5f316e7'),
        ('BasisFunc.z09', '1cf9d433-edc6-48fe-8d6e-f4d5769ed078', 1073741824, '8ad8dcd0654133e517182ed888118e40'),
    ],
}

In [ ]:
# ---- 2. Folders, log, small helpers ---------------------------------------------------------------------------
T0 = time.time()


def pick_scratch():
    """Scratch folder with the most free space. Archives live here and are deleted after parsing."""
    if os.environ.get('CS_SCRATCH'):
        return os.environ['CS_SCRATCH']
    best, free = None, -1
    for cand in (('/kaggle/temp', '/tmp', WORK) if ON_KAGGLE else ('/tmp', WORK)):
        try:
            os.makedirs(cand, exist_ok=True)
            f = shutil.disk_usage(cand).free
            if f > free:
                best, free = cand, f
        except OSError:
            pass
    return os.path.join(best, 'cs_scratch')


SCRATCH = pick_scratch()
DL = os.path.join(SCRATCH, 'downloads')       # repository files
TMP = os.path.join(SCRATCH, 'tmp')            # nested archives, extracted trees
OUT = os.path.join(SCRATCH, 'out')            # result files before packaging
CORE, EXTRA = os.path.join(OUT, 'core'), os.path.join(OUT, 'extra')
for d in (WORK, DL, TMP, CORE, EXTRA):
    os.makedirs(d, exist_ok=True)
LOG_PATH = os.path.join(CORE, 'run_log.txt')


def log(*parts):
    line = '[%6.0fs] %s' % (time.time() - T0, ' '.join(str(p) for p in parts))
    print(line, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as fh:
        fh.write(line + '\n')


def human(n):
    n = float(n or 0)
    for unit in ('B', 'kB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or unit == 'TB':
            return ('%.0f %s' if unit == 'B' else '%.1f %s') % (n, unit)
        n /= 1024


def free_gb(path):
    return shutil.disk_usage(path).free / 1024 ** 3


def hours():
    return (time.time() - T0) / 3600


def safe_name(s):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', s)[:120] or 'x'


log('notebook version', NOTEBOOK_VERSION, '| python', sys.version.split()[0], '| workers', N_WORKERS)
log('results ->', WORK, '| scratch ->', SCRATCH, '| free there: %.1f GB' % free_gb(SCRATCH))

In [ ]:
# ---- 3. Repository: list the files of a record and download them ------------------------------------------------
import requests

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'ConformalStereo-notebook/1.0 (reuse of open research data; python-requests)',
                        'Accept': 'application/json, */*'})
SKIP_BUNDLES = {'LICENSE', 'THUMBNAIL', 'TEXT', 'CC-LICENSE', 'BRANDED_PREVIEW', 'SWORD', 'METADATA', 'ORE'}
DEAD_BASES = set()                                            # API addresses that did not answer at all
DEAD_HOSTS = set()                                            # download hosts that could not be connected to


def _retry_wait(resp, attempt):
    try:
        ra = float(resp.headers.get('Retry-After', '')) if resp is not None else 0.0
    except ValueError:
        ra = 0.0
    return min(ra or 2.0 ** (attempt + 1), 120.0)


class Unreachable(RuntimeError):
    pass


def http_json(url, params=None, tries=4):
    last = 'no attempt'
    for attempt in range(tries):
        try:
            r = SESSION.get(url, params=params, timeout=(20, 90))
        except requests.RequestException as exc:
            last = repr(exc)[:200]
            if attempt + 1 < tries:
                time.sleep(min(2.0 ** (attempt + 1), 30))
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            last = 'HTTP %d' % r.status_code
            time.sleep(_retry_wait(r, attempt))
            continue
        if r.status_code != 200:
            raise RuntimeError('HTTP %d for %s: %r' % (r.status_code, r.url, r.text[:160]))
        try:
            return r.json()
        except ValueError:
            raise RuntimeError('not JSON from %s (content-type %s): %r' % (r.url, r.headers.get('Content-Type'), r.text[:160]))
    raise Unreachable('no answer from %s after %d tries: %s' % (url, tries, last))


def paged(url, key, page_size=100, max_pages=400):
    """All embedded objects of a paginated DSpace list (follows the 'next' link, or counts pages)."""
    items, seen, params, page_no = [], set(), {'size': page_size}, 0
    for _ in range(max_pages):
        mark = (url, json.dumps(params, sort_keys=True))
        if mark in seen:
            break
        seen.add(mark)
        j = http_json(url, params=params)
        items.extend((j.get('_embedded') or {}).get(key) or [])
        page = j.get('page') or {}
        number = page.get('number', page_no)
        nxt = ((j.get('_links') or {}).get('next') or {}).get('href')
        if nxt:
            url, params, page_no = nxt, None, number + 1      # 'next' links already carry the query
        elif page.get('totalPages') is not None and number + 1 < page['totalPages']:
            page_no = number + 1
            url, params = url.split('?')[0], {'size': page.get('size') or page_size, 'page': page_no}
        else:
            break
    return items


def bundle_addresses(base, dep):
    """Where the bundle list of a record may be: by item id first, then through its handle."""
    if dep.get('item'):
        yield '%s/core/items/%s/bundles' % (base, dep['item'])
    for pid in (('hdl:' + dep['handle'], dep['handle']) if dep.get('handle') else ()):
        try:
            r = SESSION.get('%s/pid/find' % base, params={'id': pid}, timeout=(20, 60))     # answers with a redirect to the item
            j = r.json() if r.status_code == 200 else {}
        except (requests.RequestException, ValueError):
            continue
        href = ((j.get('_links') or {}).get('bundles') or {}).get('href')
        uuid = j.get('uuid') or j.get('id')
        if href or uuid:
            yield href or '%s/core/items/%s/bundles' % (base, uuid)
            return


def list_record(dep):
    """Files of one record: [{name, size, md5, uuid, urls}], and how the list was obtained."""
    errors = []
    for base in API_BASES:
        if base in DEAD_BASES:
            errors.append('%s: skipped, it did not answer earlier in this run' % base)
            continue
        for address in bundle_addresses(base, dep):
            try:
                bundles = paged(address, 'bundles')
                wanted = [b for b in bundles if b.get('name') == 'ORIGINAL'] or \
                         [b for b in bundles if b.get('name') not in SKIP_BUNDLES]
                files = []
                for b in wanted:
                    href = ((b.get('_links') or {}).get('bitstreams') or {}).get('href') or \
                           '%s/core/bundles/%s/bitstreams' % (base, b.get('uuid') or b.get('id'))
                    for bs in paged(href, 'bitstreams'):
                        uuid = bs.get('uuid') or bs.get('id')
                        cs = bs.get('checkSum') or {}
                        content = ((bs.get('_links') or {}).get('content') or {}).get('href')
                        urls = [u for u in (content, '%s/core/bitstreams/%s/content' % (base, uuid),
                                            '%s/bitstreams/%s/download' % (UI_BASE, uuid)) if u]
                        files.append(dict(name=bs.get('name') or uuid, size=bs.get('sizeBytes'), uuid=uuid, bundle=b.get('name'),
                                          md5=cs.get('value') if str(cs.get('checkSumAlgorithm', '')).upper() == 'MD5' else None,
                                          urls=list(dict.fromkeys(urls))))
                if files:
                    return files, 'api:' + base
                errors.append('%s: record found but no files in bundles %s' % (address, [b.get('name') for b in bundles]))
            except Unreachable as exc:
                DEAD_BASES.add(base)
                errors.append('%s: %s' % (base, str(exc)[:300]))
                break
            except Exception as exc:                          # try the next address, then the fallback list
                errors.append('%s: %s' % (address, str(exc)[:300]))
    for e in errors:
        log('   listing failed |', e)
    known = KNOWN_FILES.get(dep['key']) or []
    files = [dict(name=k[0], size=k[2], uuid=k[1], bundle='ORIGINAL', md5=(k[3] if len(k) > 3 else None),
                  urls=['%s/core/bitstreams/%s/content' % (b, k[1]) for b in API_BASES] + ['%s/bitstreams/%s/download' % (UI_BASE, k[1])])
             for k in known]
    return files, 'fallback list (incomplete)' if files else 'none'


def md5_of(path, block=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(block), b''):
            h.update(chunk)
    return h.hexdigest()


def _fetch(url, part, pos):
    """One request. Returns (state, note, total size if the server told it, seconds to wait).
    state: 'done' = stream ended cleanly | 'retry' = transient | 'restart' = drop the partial file | 'fatal' = this address is no good."""
    headers = {'Accept': '*/*'}
    if pos:
        headers['Range'] = 'bytes=%d-' % pos
    try:
        with SESSION.get(url, headers=headers, stream=True, timeout=(20, 180)) as r:
            code = r.status_code
            if code in (429, 500, 502, 503, 504):
                return 'retry', 'HTTP %d' % code, None, _retry_wait(r, 0)
            if code == 416:
                return 'restart', 'range refused', None, 0
            if code not in (200, 206):
                return 'fatal', 'HTTP %d from %s' % (code, url), None, 0
            if 'text/html' in (r.headers.get('Content-Type') or '').lower():
                return 'fatal', 'a web page came back instead of the file from %s' % url, None, 0
            total = None
            m = re.match(r'bytes\s+\d+-\d+/(\d+)', r.headers.get('Content-Range') or '')
            if code == 206 and m:
                total = int(m.group(1))
            elif code == 200 and (r.headers.get('Content-Length') or '').isdigit() and not r.headers.get('Content-Encoding'):
                total = int(r.headers['Content-Length'])
            with open(part, 'ab' if (pos and code == 206) else 'wb') as fh:
                for chunk in r.iter_content(1 << 16):         # small chunks: a broken connection loses little
                    if chunk:
                        fh.write(chunk)
            return 'done', '', total, 0
    except (requests.RequestException, OSError) as exc:
        return 'retry', repr(exc)[:160], None, -1             # -1: the connection itself failed


def download(f, dest, tries=5):
    """Download one repository file with resume and checks. Returns (ok, note)."""
    part = dest + '.part'
    note = 'no reachable address: the repository did not answer earlier in this run'
    for url in f['urls']:
        host = '/'.join(url.split('/')[:3])
        if host in DEAD_HOSTS or any(url.startswith(b) for b in DEAD_BASES):
            continue
        if os.path.exists(part):
            os.remove(part)
        expected, stalls, md5_retry, answered = f.get('size'), 0, False, False
        for _ in range(300):
            if stalls >= tries:
                if not answered:
                    DEAD_HOSTS.add(host)                      # never connected: do not wait for this host again
                break
            pos = os.path.getsize(part) if os.path.exists(part) else 0
            state, msg, total, wait = _fetch(url, part, pos)
            answered = answered or wait != -1
            wait = max(wait, 0)
            note = msg or note
            if total and not expected:
                expected = total
            got = os.path.getsize(part) if os.path.exists(part) else 0
            stalls = 0 if got > pos else stalls + 1
            if state == 'fatal':
                break
            if state == 'restart':
                if os.path.exists(part):
                    os.remove(part)
                continue
            if state == 'retry':
                time.sleep(wait or min(2.0 ** stalls, 30))
                continue
            if expected and got < expected:
                note = 'connection closed at %s of %s' % (human(got), human(expected))
                continue                                      # resume from where it stopped
            if expected and got > expected:
                note = 'more bytes than expected'
                os.remove(part)
                continue
            if f.get('md5'):
                if md5_of(part) != f['md5'].lower():
                    note = 'checksum mismatch'
                    os.remove(part)
                    if md5_retry:
                        break
                    md5_retry = True
                    continue
                check = 'md5 ok'
            else:
                check = 'size ok' if expected else 'no size or checksum to compare'
            os.replace(part, dest)
            return True, check
    if os.path.exists(part):
        os.remove(part)
    return False, note


def local_index():
    """Files already on disk under the input folders (a Kaggle dataset uploaded by hand), by file name."""
    idx = {}
    for root in INPUT_DIRS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath[len(root):].count(os.sep) >= 5:
                dirnames[:] = []
            for fn in filenames:
                idx.setdefault(fn, os.path.join(dirpath, fn))
    return idx

In [ ]:
# ---- 4. Gaussian output parser --------------------------------------------------------------------------------
try:
    import numpy as np
except Exception:                                             # fingerprints are skipped without numpy
    np = None

PERIODIC = ('X H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr '
            'Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt '
            'Au Hg Tl Pb Bi Po At Rn').split()
SYMBOL_Z = {s: i for i, s in enumerate(PERIODIC)}

# Every pattern starts with literal text, so the search runs at C speed on outputs of hundreds of megabytes.
G_ROUTE = re.compile(rb'\n -{5,}[ \t]*\n((?: #[^\n]*\n)(?: [^\n]*\n)*?) -{5,}[ \t]*\n')
G_VER = re.compile(rb'Gaussian (\d\d):\s+(\S+)')
G_SCF = re.compile(rb'SCF Done:\s+E\((\S+?)\)\s*=\s*(-?\d+\.\d+(?:[DEde][+-]?\d+)?)\s+A\.U\. after\s+(\d+) cycles')
G_ISO = re.compile(rb'\n[ \t]*(\d+)[ \t]+([A-Za-z]{1,3})[ \t]+Isotropic =[ \t]*(-?\d*\.\d+|\*+)[ \t]+Anisotropy =[ \t]*(-?\d*\.\d+|\*+)')
G_XYZ = re.compile(rb'[ \t]*(\d+)[ \t]+(-?\d+)[ \t]+(-?\d+)[ \t]+(-?\d+\.\d+)[ \t]+(-?\d+\.\d+)[ \t]+(-?\d+\.\d+)[ \t]*\n')
G_CHG = re.compile(rb'Charge\s*=\s*(-?\d+)\s+Multiplicity\s*=\s*(\d+)')
G_NAT = re.compile(rb'NAtoms=\s*(\d+)')
G_SOLV = re.compile(rb'Solvent\s*:\s*([^,\n]+),\s*Eps\s*=\s*(\d+\.\d+)')
G_CPU = re.compile(rb'Job cpu time:\s+(\d+) days\s+(\d+) hours\s+(\d+) minutes\s+([\d.]+) seconds')
G_NPROC = re.compile(rb'Will use up to\s+(\d+) processors')
G_GIBBS = re.compile(rb'Sum of electronic and thermal Free Energies=\s*(-?\d+\.\d+)')
G_ZPE = re.compile(rb'Sum of electronic and zero-point Energies=\s*(-?\d+\.\d+)')
G_FREQ = re.compile(rb'\n Frequencies --((?:[ \t]+-?\d+\.\d+)+)')


def _f(b):
    return float(b.replace(b'D', b'E').replace(b'd', b'e'))


def sniff_program(blob):
    head = blob[:60000]
    if b'Entering Gaussian System' in head or b'Gaussian, Inc.' in head or G_VER.search(head):
        return 'gaussian'
    if b'Northwest Computational Chemistry Package' in head:
        return 'nwchem'
    if b'O   R   C   A' in head:
        return 'orca'
    if b'BatchMin' in head or b'MacroModel' in head or b'Schrodinger' in head or b'Schr\xc3\xb6dinger' in head:
        return 'macromodel'
    if b'TINKER' in head or b'Tinker' in head:
        return 'tinker'
    if b'Jaguar' in head:
        return 'jaguar'
    return 'unknown'


HARVEST_RE = re.compile(r'isotropic|\biso\s*=|total dft energy|total scf energy|scfe:|final single point energy|nmr properties for atom|'
                        r'^\s*atom:\s+\d+|^\s*nucleus\s+\d+|total energy\s*=|shielding tensor', re.I | re.M)


def harvest_lines(blob, limit=700, width=140):
    """Lines that carry energies or shieldings in the output of a program this notebook has no reader for.
    They are kept as text, so that such outputs can be read later without downloading the deposit again."""
    text = blob.decode('latin-1')
    out = []
    for m in HARVEST_RE.finditer(text):
        a = text.rfind('\n', 0, m.start()) + 1
        b = text.find('\n', m.end())
        line = text[a:b if b >= 0 else len(text)].strip()[:width]
        if not out or out[-1] != line:
            out.append(line)
        if len(out) >= limit:
            break
    return out


def moments(xyz, z=None):
    """Principal moments of the point set (unit masses), all atoms and heavy atoms: a rotation-free conformer fingerprint."""
    if np is None or len(xyz) < 9:
        return None, None
    a = np.asarray(xyz, float).reshape(-1, 3)
    out = []
    for sel in (None, [i for i, zz in enumerate(z or []) if zz and zz > 1]):
        b = a if sel is None else (a[sel] if len(sel) >= 3 else None)
        if b is None:
            out.append(None)
            continue
        c = b - b.mean(axis=0)
        r2 = (c * c).sum()
        w = np.linalg.eigvalsh(r2 * np.eye(3) - c.T @ c)
        out.append([round(float(v), 3) for v in w])
    return out[0], out[1]


def _orientations(blob, lo, hi):
    """Positions and kinds of the geometry tables between lo and hi."""
    out = []
    i = blob.find(b' orientation:', lo, hi)
    while i >= 0:
        w = blob[max(lo, i - 8):i]
        kind = 'Standard' if w.endswith(b'Standard') else 'Input' if w.endswith(b'Input') else 'Z-Matrix' if w.endswith(b'Z-Matrix') else None
        if kind:
            out.append((i, kind))
        i = blob.find(b' orientation:', i + 13, hi)
    return out


def _orientation(blob, pos, hi):
    """Rows of the geometry table whose title is at pos: (atomic numbers, flat xyz)."""
    p = pos
    for _ in range(5):                                        # title line, rule, two header lines, rule
        q = blob.find(b'\n', p, hi)
        if q < 0:
            return [], []
        p = q + 1
    z, xyz = [], []
    while True:
        m = G_XYZ.match(blob, p, hi)
        if not m:
            break
        z.append(int(m.group(2)))
        xyz.extend((round(float(m.group(4)), 4), round(float(m.group(5)), 4), round(float(m.group(6)), 4)))
        p = m.end()
    return z, xyz


def _last(pattern, blob, lo, hi):
    m = None
    for m in pattern.finditer(blob, lo, hi):
        pass
    return m


# Spin-spin coupling constants. Every title that ends in "K (Hz):" or "J (Hz):" is followed by a lower triangle that is
# printed five columns at a time:
#  Total nuclear spin-spin coupling J (Hz):
#                 2             4             6             8             9
#       2  0.000000D+00
#       4  0.926067D+01  0.000000D+00
# The numbers at the start of the rows are atom numbers; with NMR=ReadAtoms only the chosen atoms are printed.
G_JTITLE = re.compile(rb'([A-Za-z][^\n]{2,70}?)[ \t]+([KJ])[ \t]*\(Hz\)[ \t]*:?')
G_JNUM = re.compile(rb'[-+]?(?:\d+\.\d*|\.\d+)(?:[DEde][-+]?\d+)?')
J_NAMES = (('fermi', 'FC'), ('spin-dipolar', 'SD'), ('paramagnetic', 'PSO'), ('diamagnetic', 'DSO'), ('total', 'total'))


def _triangle(blob, pos, hi):
    """Lower triangle that starts at pos. Returns (row labels, values, layout as expected, end, every pair read).
    Values run row by row, each row up to and including its diagonal element, as the matrix reads when put together."""
    cols, allcols, order, seen, vals = None, [], [], set(), {}
    p, blank = pos, 0
    while p < hi:
        q = blob.find(b'\n', p, hi)
        if q < 0:
            q = hi
        line = blob[p:q]
        toks = line.split()
        if not toks:
            blank += 1
            if cols is not None or blank > 2:
                break
        elif all(t.isdigit() for t in toks):                  # the numbers of the next (up to) five columns
            cols = [int(t) for t in toks]
            allcols.extend(cols)
        elif cols is not None and toks[0].isdigit():
            rest = line.split(None, 1)[1]
            nums = G_JNUM.findall(rest)
            if not nums or len(nums) > len(cols) or G_JNUM.sub(b'', rest).strip():
                break
            r = int(toks[0])
            if r not in seen:
                seen.add(r)
                order.append(r)
            for c, x in zip(cols, nums):
                vals[(r, c)] = _f(x)
        else:
            break
        p = q + 1
    n = len(order)
    # complete only if the column numbers of all blocks, in the order printed, are the row numbers: a matrix cut short fails this
    good = n > 0 and allcols == order and len(vals) == n * (n + 1) // 2 and \
        all((order[i], order[j]) in vals for i in range(n) for j in range(i + 1))
    flat = [vals[(order[i], order[j])] for i in range(n) for j in range(i + 1)] if good else None
    return order, flat, good, min(p, hi), vals


def spin_spin(blob, lo, hi, want_raw=False):
    """Coupling matrices of one job step, or None if the step prints none. Keys of the result:
    jt = every matrix title of the step in the order printed (short name:K or :J);
    J = {name: {lab: atom numbers, v: lower triangle row by row}} for the J matrices named in J_TERMS;
    bad = names whose layout was not the expected one (their J entry then holds p = [row, column, value] triples);
    raw = {name: the text as printed}, for the bad ones and, if want_raw, for all."""
    out = None
    known = [short for _, short in J_NAMES]
    p = blob.find(b'(Hz)', lo, hi)
    while p >= 0:
        a = max(lo, blob.rfind(b'\n', lo, p) + 1)
        b = blob.find(b'\n', p, hi)
        if b < 0:
            b = hi
        nxt = b
        m = G_JTITLE.fullmatch(blob[a:b].strip())
        if m:
            if out is None:
                out = dict(jt=[], J={}, bad=[], raw={})
            title = m.group(1).decode('latin-1', 'replace')
            name = next((short for key, short in J_NAMES if key in title.lower()), title[:40])
            kind = m.group(2).decode('latin-1')
            out['jt'].append(name + ':' + kind)
            if kind == 'J' and (name in J_TERMS or name not in known):
                lab, flat, good, end, vals = _triangle(blob, min(b + 1, hi), hi)
                key = name
                while key in out['J']:                        # the same title twice in one job step
                    key += '+'
                if good:
                    out['J'][key] = dict(lab=lab, v=flat)
                else:
                    out['bad'].append(key)
                    out['J'][key] = dict(lab=lab, p=[[r, c, x] for (r, c), x in sorted(vals.items())])
                if want_raw or not good:
                    stop = blob.find(b'\n', end, hi)          # one line past the matrix, to show what ended it
                    out['raw'][key] = blob[a:(stop if stop >= 0 else hi)][:400000].decode('latin-1', 'replace')
                nxt = max(b, end - 1)
        p = blob.find(b'(Hz)', nxt, hi)
    return out


def parse_gaussian(blob, want_j=False, want_raw=False):
    """One record per job step (route section) of a Gaussian output. Returns (core records, extra records).
    With want_j, a step that prints coupling matrices carries them under the key '_j' (see spin_spin)."""
    if b'\r\n' in blob:
        blob = blob.replace(b'\r\n', b'\n')
    n = len(blob)
    ver = G_VER.search(blob, 0, min(n, 200000))
    nproc = G_NPROC.search(blob, 0, min(n, 200000))
    routes = list(G_ROUTE.finditer(blob))
    starts = [m.start() for m in routes] or [0]
    tail = blob[-400:].rstrip().split(b'\n')[-1].decode('latin-1', 'replace').strip()[:100]
    core, extra = [], []
    for k, lo in enumerate(starts):
        hi = starts[k + 1] if k + 1 < len(starts) else n
        rec = {'job': k, 'njobs': len(starts)}
        if ver:
            rec['ver'] = (ver.group(1) + b' ' + ver.group(2)).decode('latin-1')
        if nproc:
            rec['nproc'] = int(nproc.group(1))
        if routes:
            lines = routes[k].group(1).split(b'\n')
            rec['route'] = re.sub(r'\s+', ' ', ''.join(ln[1:].decode('latin-1', 'replace') for ln in lines if ln)).strip()[:600]
        m = G_CHG.search(blob, lo, hi)
        if m:
            rec['chg'], rec['mult'] = int(m.group(1)), int(m.group(2))
        m = G_NAT.search(blob, lo, hi)
        if m:
            rec['nat'] = int(m.group(1))
        i = blob.find(b' basis functions,', lo, hi)
        if i >= 0:
            word = blob[max(lo, i - 12):i].split()
            if word and word[-1].isdigit():
                rec['nbf'] = int(word[-1])
        if blob.find(b'Solvent', lo, hi) >= 0:
            m = G_SOLV.search(blob, lo, hi)
            if m:
                rec['solv'], rec['eps'] = m.group(1).decode('latin-1').strip(), float(m.group(2))
        scf, n_scf = None, 0
        for scf in G_SCF.finditer(blob, lo, hi):
            n_scf += 1
        if scf is not None:
            rec['scf'], rec['scf_m'], rec['n_scf'], rec['scf_cyc'] = _f(scf.group(2)), scf.group(1).decode('latin-1'), n_scf, int(scf.group(3))
        if blob.find(b'Sum of electronic', lo, hi) >= 0:
            m = _last(G_GIBBS, blob, lo, hi)
            if m is not None:
                rec['gibbs'] = float(m.group(1))
            m = _last(G_ZPE, blob, lo, hi)
            if m is not None:
                rec['e_zpe'] = float(m.group(1))
        if blob.find(b' Frequencies --', lo, hi) >= 0:
            freqs = [float(x) for m in G_FREQ.finditer(blob, lo, hi) for x in m.group(1).split()]
            if freqs:
                rec['n_freq'], rec['n_imag'] = len(freqs), sum(1 for x in freqs if x < 0)
        rec['norm'] = blob.count(b'Normal termination of Gaussian', lo, hi)
        if blob.find(b'Error termination', lo, hi) >= 0:
            rec['err'] = 1
        if blob.find(b'Optimization completed', lo, hi) >= 0:
            rec['opt_done'] = 1
        elif blob.find(b'Optimization stopped', lo, hi) >= 0 or blob.find(b'Number of steps exceeded', lo, hi) >= 0:
            rec['opt_done'] = 0
        cpu = sum(int(m.group(1)) * 86400 + int(m.group(2)) * 3600 + int(m.group(3)) * 60 + float(m.group(4)) for m in G_CPU.finditer(blob, lo, hi))
        if cpu:
            rec['cpu_s'] = round(cpu, 1)
        if k == len(starts) - 1:
            rec['tail'] = tail
        # NMR shieldings: the last tensor table of this step
        h = blob.rfind(b'Magnetic shielding tensor (ppm)', lo, hi)
        el = iso = aniso = None
        if h >= 0:
            el, iso, aniso = [], [], []
            for m in G_ISO.finditer(blob, h, hi):
                if int(m.group(1)) != len(el) + 1:
                    break
                el.append(m.group(2).decode('latin-1'))
                iso.append(None if m.group(3).startswith(b'*') else float(m.group(3)))
                aniso.append(None if m.group(4).startswith(b'*') else float(m.group(4)))
            rec['iso'] = iso
        # geometries: first table as given, last table (standard orientation if printed)
        oris = _orientations(blob, lo, hi)
        xrec = {'job': k}
        if oris:
            z1, xyz1 = _orientation(blob, oris[0][0], hi)
            std = [o for o in oris if o[1] == 'Standard'] or [o for o in oris if o[1] == oris[0][1]]
            n_geom = max(len([o for o in oris if o[1] == kind]) for kind in ('Input', 'Standard', 'Z-Matrix'))
            rec['n_geom'] = n_geom
            if z1:
                rec.setdefault('nat', len(z1))
                rec['pm'], rec['pmh'] = moments(xyz1, z1)
                xrec.update(z=z1, first_kind=oris[0][1], first=xyz1)
                if el is None:
                    el = [PERIODIC[v] if 0 <= v < len(PERIODIC) else 'X' for v in z1]
            if n_geom > 1:                                    # an optimisation: keep the final geometry as well
                z2, xyz2 = _orientation(blob, std[-1][0], hi)
                if z2:
                    rec['pm_last'], rec['pmh_last'] = moments(xyz2, z2)
                    xrec.update(last_kind=std[-1][1], last=xyz2)
        if el:
            rec['el'] = ' '.join(el)
        if aniso:
            xrec['aniso'] = aniso
        if want_j and blob.find(b'(Hz)', lo, hi) >= 0:
            js = spin_spin(blob, lo, hi, want_raw)
            if js is not None:
                rec['nj'], rec['_j'] = len(js['J']), js
        core.append({a: b for a, b in rec.items() if b is not None})
        if len(xrec) > 1:
            extra.append(xrec)
    return core, extra


def job_class(route):
    r = (route or '').lower()
    if 'nmr' in r:
        return 'nmr'
    if re.search(r'\bopt\b|opt=|opt\(', r):
        return 'opt+freq' if 'freq' in r else 'opt'
    if 'freq' in r:
        return 'freq'
    return 'sp' if r else 'no route'

In [ ]:
# ---- 5. Maestro (.mae) parser ---------------------------------------------------------------------------------
# Two independent readers. The first works on tokens and reads a file as a stream; the second works line by line,
# the way the original PyDP4 reader does, and is used when the first one stops. Both accept full (f_m_ct) and
# partial (p_m_ct) structures: rows are read up to the closing ':::', with or without the index column, and a
# partial structure that lists only some atoms takes the others from the last full structure.
MAE_TOK = re.compile(r'"(?:[^"\\]|\\.)*"|#[^#\n]*#|[{}]|:::|[^\s{}]+')
MAE_CT = re.compile(r'^[ \t]*([fp])_m_ct[ \t]*\{', re.M)
MAE_IDX = re.compile(r'^(\w+)\[(\d+)\]$')
MAE_ROW = re.compile(r'"(?:[^"\\]|\\.)*"|\S+')


def _mae_val(name, tok):
    if tok.startswith('"') and tok.endswith('"') and len(tok) >= 2:
        tok = tok[1:-1].replace('\\"', '"').replace('\\\\', '\\')
    if tok in ('<>', ''):
        return None
    try:
        if name.startswith('r_'):
            return float(tok)
        if name.startswith(('i_', 'b_')):
            return int(tok)
    except ValueError:
        pass
    return tok[:200]


def _mae_rows(body, cols, declared):
    """Cut the tokens of a table body into rows. Returns rows that always start with the row index (as text)."""
    nc = len(cols)
    if nc == 0 or not body:
        return []
    def is_index(seq):
        try:
            v = [int(x) for x in seq]
        except ValueError:
            return False
        return all(b > a for a, b in zip(v, v[1:])) and v[0] >= 1
    if len(body) % (nc + 1) == 0 and is_index(body[0::nc + 1]) and (len(body) // (nc + 1) == declared or len(body) % nc != 0 or not declared or len(body) // nc != declared):
        return [body[k:k + nc + 1] for k in range(0, len(body), nc + 1)]
    if len(body) % nc == 0:
        return [[str(k // nc + 1)] + body[k:k + nc] for k in range(0, len(body), nc)]
    if len(body) % (nc + 1) == 0:
        return [body[k:k + nc + 1] for k in range(0, len(body), nc + 1)]
    raise ValueError('table with %d values does not fit %d columns' % (len(body), nc))


def _mae_block(toks, i):
    """Parse one block starting at its opening brace. Returns (properties, {sub-block name: (columns, rows)}, next index)."""
    if toks[i] != '{':
        raise ValueError('block does not start with a brace')
    i += 1
    names = []
    while toks[i] != ':::' and toks[i] != '}' and not (toks[i + 1] == '{'):
        names.append(toks[i])
        i += 1
    props = {}
    if toks[i] == ':::':
        i += 1
        props = {n: _mae_val(n, toks[i + k]) for k, n in enumerate(names)}
        i += len(names)
    subs = {}
    while toks[i] != '}':
        name = toks[i]
        i += 1
        m = MAE_IDX.match(name)
        if m:
            if toks[i] != '{':
                raise ValueError('indexed block %s without a brace' % name)
            i += 1
            cols = []
            while toks[i] != ':::':
                cols.append(toks[i])
                i += 1
            i += 1
            j = i
            while toks[j] != ':::' and toks[j] != '}' and toks[j] != '{':
                j += 1
            rows = _mae_rows(toks[i:j], cols, int(m.group(2)))
            i = j
            if toks[i] == ':::':
                i += 1
            while toks[i] != '}':                             # nested indexed sub-blocks are skipped
                if toks[i] == '{':
                    depth = 1
                    i += 1
                    while depth:
                        depth += {'{': 1, '}': -1}.get(toks[i], 0)
                        i += 1
                else:
                    i += 1
            i += 1
            subs[m.group(1)] = (cols, rows, int(m.group(2)))
        else:
            _, _, i = _mae_block(toks, i)
    return props, subs, i + 1


def _mae_finish(kind, props, atom_table, state):
    """Common end of both readers: one structure as a dict. state carries the last full structure (atoms, coordinates)."""
    ct = {'full': kind == 'f', 'props': props}
    if atom_table is not None:
        cols, rows, declared = atom_table
        ix = {c: j + 1 for j, c in enumerate(cols)}
        n_decl = declared or len(rows)
        base = state.get('xyz') if (kind == 'p' and state.get('xyz') is not None and len(state['xyz']) == 3 * n_decl) else None
        if all(c in ix for c in ('r_m_x_coord', 'r_m_y_coord', 'r_m_z_coord')) and rows:
            if len(rows) == n_decl:
                xyz = []
                for r in rows:
                    xyz.extend((round(float(r[ix['r_m_x_coord']]), 4), round(float(r[ix['r_m_y_coord']]), 4), round(float(r[ix['r_m_z_coord']]), 4)))
                ct['xyz'] = xyz
            elif base is not None and len(rows) < n_decl:   # partial structure that lists only the atoms that moved
                xyz = list(base)
                for r in rows:
                    a = int(r[0]) - 1
                    if not 0 <= a < n_decl:
                        raise ValueError('atom index %s outside the structure' % r[0])
                    xyz[3 * a:3 * a + 3] = (round(float(r[ix['r_m_x_coord']]), 4), round(float(r[ix['r_m_y_coord']]), 4), round(float(r[ix['r_m_z_coord']]), 4))
                ct['xyz'] = xyz
                ct['merged'] = len(rows)
            else:
                raise ValueError('%d coordinate rows for %d atoms' % (len(rows), n_decl))
        if 'i_m_atomic_number' in ix and len(rows) == n_decl:
            state['z'] = [int(r[ix['i_m_atomic_number']]) for r in rows]
        if 'i_m_formal_charge' in ix and len(rows) == n_decl:
            ct['charge'] = sum(int(r[ix['i_m_formal_charge']]) for r in rows)
        if 'i_m_mmod_type' in ix and 'i_m_atomic_number' not in ix and len(rows) == n_decl:
            ct['mmod_types'] = [int(r[ix['i_m_mmod_type']]) for r in rows]
            state['mmod_types'] = ct['mmod_types']
        ct['n_atoms'] = n_decl
        if kind == 'f' and 'xyz' in ct:
            state['xyz'] = ct['xyz']
        if state.get('z') is not None and len(state['z']) == n_decl:
            ct['z'] = state['z']
        elif 'mmod_types' not in ct and state.get('mmod_types') is not None and len(state['mmod_types']) == n_decl:
            ct['mmod_types'] = state['mmod_types']
    return ct


def _mae_ct(kind, chunk, state):
    """One structure block (text starting at its opening brace) -> dict."""
    toks = [t for t in MAE_TOK.findall(chunk) if not (t.startswith('#') and t.endswith('#') and len(t) > 1)]
    toks.append('}')                                          # guard: a cut block ends the scan instead of running off the list
    props, subs, _ = _mae_block(toks, 0)
    return _mae_finish(kind, props, subs.get('m_atom'), state)


def iter_mae(fh, block=8 << 20):
    """Structures of a Maestro file, read from a binary stream one at a time (partial structures inherit the atom list)."""
    buf, state, pending = '', {}, None
    while True:
        raw = fh.read(block)
        buf += raw.decode('latin-1')
        marks = list(MAE_CT.finditer(buf))
        if pending is None and marks:
            buf = buf[marks[0].start():]
            marks = list(MAE_CT.finditer(buf))
            pending = True
        limit = len(marks) if not raw else len(marks) - 1     # the last block may continue in the next read
        for k in range(max(limit, 0)):
            end = marks[k + 1].start() if k + 1 < len(marks) else len(buf)
            yield _mae_ct(marks[k].group(1), buf[marks[k].end() - 1:end], state)
        if not raw:
            return
        if marks:
            buf = buf[marks[-1].start():]
        elif len(buf) > 4096:
            buf = buf[-4096:]                                 # header before the first structure: keep only a tail


def iter_mae_lines(text):
    """Second reader, line by line (the logic of the original PyDP4 reader): used when the token reader stops."""
    lines = text.replace('\r', '').split('\n')
    starts = [(i, m.group(1)) for i, m in ((i, MAE_CT.match(l)) for i, l in enumerate(lines)) if m]
    state = {}
    for b, (start, kind) in enumerate(starts):
        end = starts[b + 1][0] if b + 1 < len(starts) else len(lines)
        blk = lines[start + 1:end]
        k, names = 0, []
        while k < len(blk) and blk[k].strip() != ':::' and not re.match(r'^\s*m_\w+(\[\d+\])?\s*\{', blk[k]):
            s = blk[k].strip()
            if s and not s.startswith('#'):
                names.append(s)
            k += 1
        props = {}
        if k < len(blk) and blk[k].strip() == ':::':
            k += 1
            rest, vals, pos = '\n'.join(blk[k:]), [], 0       # a quoted value may run over several lines
            for m in MAE_ROW.finditer(rest):
                if len(vals) >= len(names):
                    break
                vals.append(m.group(0))
                pos = m.end()
            k += rest[:pos].count('\n') + (1 if vals else 0)
            props = {n: _mae_val(n, v) for n, v in zip(names, vals)}
        table = None
        for a in range(k, len(blk)):
            m = re.match(r'^\s*m_atom\[(\d+)\]\s*\{', blk[a])
            if not m:
                continue
            cols, r = [], a + 1
            while r < len(blk) and blk[r].strip() != ':::':
                s = blk[r].strip()
                if s and not s.startswith('#'):
                    cols.append(s)
                r += 1
            body, r = [], r + 1
            while r < len(blk) and blk[r].strip() not in (':::', '}'):
                body.extend(MAE_ROW.findall(blk[r]))
                r += 1
            table = (cols, _mae_rows(body, cols, int(m.group(1))), int(m.group(1)))
            break
        yield _mae_finish(kind, props, table, state)


def parse_mae(text):
    """Structures of a Maestro file given as text. Returns a list of dicts: props, n_atoms, z, xyz, charge, full."""
    return list(iter_mae(io.BytesIO(text.encode('latin-1'))))


def mae_diag(text, n=42, width=150):
    """Lines around the first two structure blocks of a Maestro file: kept when a file cannot be read, to see why."""
    lines = text.replace('\r', '').split('\n')
    starts = [i for i, l in enumerate(lines) if MAE_CT.match(l)]
    out = ['lines: %d | structure blocks: %d (full %d, partial %d)' % (len(lines), len(starts), sum('f_m_ct' in lines[i] for i in starts), sum('p_m_ct' in lines[i] for i in starts))]
    for s in starts[:2] + starts[-1:] if len(starts) > 2 else starts[:2]:
        out.append('--- block at line %d' % (s + 1))
        out.extend(l[:width] for l in lines[s:s + n])
        atom = next((i for i in range(s, min(len(lines), s + 4000)) if 'm_atom[' in lines[i]), None)
        if atom is not None and atom >= s + n:
            out.append('--- its atom table at line %d' % (atom + 1))
            out.extend(l[:width] for l in lines[atom:atom + 14])
        stop = next((i for i in range((atom or s) + 1, min(len(lines), (atom or s) + 4000)) if lines[i].strip() == ':::' and i > (atom or s) + 12), None)
        if stop is not None:
            out.append('--- end of that table at line %d' % (stop + 1))
            out.extend(l[:width] for l in lines[max(stop - 3, 0):stop + 6])
    return '\n'.join(out)


def mae_energy_key(props):
    """The property that holds the conformer energy (kJ/mol in MacroModel files)."""
    keys = [k for k in props if k.startswith('r_mmod_Potential_Energy')] or \
           [k for k in props if k.startswith('r_') and 'energy' in k.lower() and 'relative' not in k.lower()]
    return keys[0] if keys else None

In [ ]:
# ---- 6. Reading archives and sorting their members --------------------------------------------------------------
QM_EXT = ('.out', '.log', '.nwo', '.nwout', '.jagout', '.orcaout')   # outputs of quantum chemistry programs (Gaussian is parsed, others are harvested)
TEXT_EXT = ('.sdf', '.mol', '.mol2', '.txt', '.dp4', '.csv', '.tsv', '.json', '.md', '.xml', '.nmr', '.py', '.cfg', '.ini',
            '.yaml', '.yml', '.dat', '.xyz', '.smi', '.inchi', '.cdxml', '.html', '.readme')
BRUKER_DIRS = ('pdata', 'proton', 'carbon')
BRUKER_FILES = {'fid', 'ser', 'acqu', 'acqus', 'acqu2', 'acqu2s', 'proc', 'procs', 'proc2', 'proc2s', '1r', '1i', '2rr', 'title',
                'pulseprogram', 'audita.txt', 'auditp.txt', 'format.temp', 'scon2', 'uxnmr.par', 'uxnmr.info', 'outd', 'peaklist.xml',
                'specpar', 'precom.output', 'shimvalues', 'cpdprg2', 'vtc_pid_settings', 'prosol_history', 'thumb.png'}


def archive_kind(name):
    n = name.lower()
    if n.endswith('.zip'):
        return 'zip'
    if n.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tbz2', '.tar.xz', '.txz')):
        return 'tar'
    if n.endswith(('.7z', '.rar')):
        return '7z'
    return None


def split_ext(path):
    """(lower-case extension without a trailing .gz, gz flag)."""
    low = path.rsplit('/', 1)[-1].lower()
    if low.endswith('.tar.gz'):
        return '.tar.gz', False
    gz = low.endswith('.gz')
    if gz:
        low = low[:-3]
    return (os.path.splitext(low)[1] if '.' in low else ''), gz


def classify(path, size):
    low = path.lower()
    base = low.rsplit('/', 1)[-1]
    comps = low.split('/')[:-1]
    ext, _ = split_ext(path)
    if archive_kind(base):
        return 'archive'
    if base.endswith(('.mae', '.maegz', '.mae.gz')):
        return 'mae'
    if ext in QM_EXT:
        return 'qm'
    bruker = base in BRUKER_FILES or any(c in BRUKER_DIRS or 'nmr_folder' in c or 'nmr_data' in c for c in comps)
    if size <= SMALL_FILE_KB * 1024 and not bruker:
        if ext in TEXT_EXT:
            return 'small'
        if ext == '' and size > 0:                            # NMR descriptions have no extension (XNMR, SX_NMR)
            return 'small'
    return 'other'


class ZipSource:
    """Members of a zip file. Each worker process opens its own handle."""
    def __init__(self, path, relax=False):
        self.path = path
        self.relax = relax                                    # only for a set joined by the zip program, see open()

    def members(self):
        with zipfile.ZipFile(self.path) as zf:
            return [(i.filename, i.file_size) for i in zf.infolist() if not i.is_dir()]

    def __getstate__(self):
        return {'path': self.path, 'relax': self.relax}

    def open(self):
        self._zf = zipfile.ZipFile(self.path)
        if self.relax:
            # Python 3.12+ refuses some ZIP64 members of a joined split set as "overlapped entries" although the
            # archive is sound; the check is switched off for that one file. Every member is still CRC-checked.
            for info in self._zf.infolist():
                if hasattr(info, '_end_offset'):
                    info._end_offset = None
        return self

    def close(self):
        zf = self.__dict__.pop('_zf', None)
        if zf is not None:
            zf.close()

    def stream(self, name):
        return self._zf.open(name)


class DirSource:
    """Files of a folder tree (an extracted tar, or a dataset uploaded by hand)."""
    def __init__(self, root):
        self.root = root

    def members(self):
        out = []
        for dirpath, _, filenames in os.walk(self.root):
            for fn in filenames:
                p = os.path.join(dirpath, fn)
                try:
                    out.append((os.path.relpath(p, self.root).replace(os.sep, '/'), os.path.getsize(p)))
                except OSError:
                    pass
        return sorted(out)

    def open(self):
        return self

    def close(self):
        pass

    def stream(self, name):
        return open(os.path.join(self.root, *name.split('/')), 'rb')


class VolumeFile:
    """The volumes of a split archive read as one file (name.z01, name.z02, ..., name.zip in that order)."""
    def __init__(self, paths):
        self.paths = list(paths)
        self.sizes = [os.path.getsize(p) for p in self.paths]
        self.starts = [sum(self.sizes[:k]) for k in range(len(self.sizes))]
        self.total = sum(self.sizes)
        self.pos, self._k, self._fh = 0, None, None

    def seek(self, pos):
        self.pos = pos

    def read(self, n):
        out = []
        while n > 0 and self.pos < self.total:
            k = max(i for i, st in enumerate(self.starts) if st <= self.pos)
            if k != self._k:
                if self._fh is not None:
                    self._fh.close()
                self._fh, self._k = open(self.paths[k], 'rb'), k
            self._fh.seek(self.pos - self.starts[k])
            chunk = self._fh.read(min(n, self.sizes[k] - (self.pos - self.starts[k])))
            if not chunk:
                break
            out.append(chunk)
            self.pos += len(chunk)
            n -= len(chunk)
        return b''.join(out)

    def close(self):
        if self._fh is not None:
            self._fh.close()
        self._fh = self._k = None


def split_zip_directory(vf):
    """Central directory of a split (multi-volume) zip, ZIP64 included. Returns a list of members with the absolute
    position of each local header in the joined volumes."""
    import struct
    tail_len = min(vf.total, (1 << 16) + 200)
    vf.seek(vf.total - tail_len)
    tail = vf.read(tail_len)
    e = tail.rfind(b'PK\x05\x06')
    if e < 0:
        raise RuntimeError('no end-of-directory record in the last volume')
    disk, cd_disk, n_here, n_tot, cd_size, cd_off, _ = struct.unpack('<HHHHIIH', tail[e + 4:e + 22])
    loc = tail.rfind(b'PK\x06\x07', 0, e)
    if loc >= 0 and (0xFFFF in (disk, cd_disk, n_here, n_tot) or 0xFFFFFFFF in (cd_size, cd_off)):
        z_disk, z_off, _ = struct.unpack('<IQI', tail[loc + 4:loc + 20])
        vf.seek(vf.starts[z_disk] + z_off)
        rec = vf.read(56)
        if rec[:4] != b'PK\x06\x06':
            raise RuntimeError('ZIP64 end-of-directory record not found where the locator points')
        _, _, _, disk, cd_disk, n_here, n_tot, cd_size, cd_off = struct.unpack('<QHHIIQQQQ', rec[4:56])
    if cd_disk >= len(vf.starts):
        raise RuntimeError('directory is on volume %d but only %d volumes are present' % (cd_disk + 1, len(vf.starts)))
    vf.seek(vf.starts[cd_disk] + cd_off)
    cd = vf.read(cd_size)
    out, i = [], 0
    while i + 46 <= len(cd) and cd[i:i + 4] == b'PK\x01\x02':
        (_, _, flags, method, _, _, crc, csize, usize, n_name, n_extra, n_comment, d_start, _, _, off) = struct.unpack('<HHHHHHIIIHHHHHII', cd[i + 4:i + 46])
        name = cd[i + 46:i + 46 + n_name]
        extra = cd[i + 46 + n_name:i + 46 + n_name + n_extra]
        j, uname = 0, None
        while j + 4 <= len(extra):
            tag, ln = struct.unpack('<HH', extra[j:j + 4])
            if tag == 0x7075 and ln > 5:                      # Info-ZIP Unicode path: the name in UTF-8
                try:
                    uname = extra[j + 9:j + 4 + ln].decode('utf-8')
                except UnicodeDecodeError:
                    uname = None
            if tag == 1:                                      # ZIP64: only the fields that overflowed, in this order
                body, q = extra[j + 4:j + 4 + ln], 0
                if usize == 0xFFFFFFFF:
                    usize, q = struct.unpack('<Q', body[q:q + 8])[0], q + 8
                if csize == 0xFFFFFFFF:
                    csize, q = struct.unpack('<Q', body[q:q + 8])[0], q + 8
                if off == 0xFFFFFFFF:
                    off, q = struct.unpack('<Q', body[q:q + 8])[0], q + 8
                if d_start == 0xFFFF:
                    d_start = struct.unpack('<I', body[q:q + 4])[0]
            j += 4 + ln
        if d_start >= len(vf.starts):
            raise RuntimeError('member on volume %d but only %d volumes are present' % (d_start + 1, len(vf.starts)))
        if flags & 0x800:
            text = name.decode('utf-8', 'replace')
        elif uname:
            text = uname
        else:                                                 # archives written on Unix hold UTF-8 names without the flag
            try:
                text = name.decode('utf-8')
            except UnicodeDecodeError:
                text = name.decode('cp437')
        out.append(dict(name=text, method=method, crc=crc, csize=csize, usize=usize,
                        start=vf.starts[d_start] + off, encrypted=bool(flags & 1)))
        i += 46 + n_name + n_extra + n_comment
    if len(out) != n_tot:
        raise RuntimeError('directory lists %d members, %d were read' % (n_tot, len(out)))
    return out


class SplitMember:
    """One member of a split zip as a readable stream (stored or deflated), checked against its CRC at the end."""
    def __init__(self, vf, entry):
        import struct, zlib
        self.vf, self.e = vf, entry
        if entry['encrypted'] or entry['method'] not in (0, 8):
            raise RuntimeError('member is encrypted or uses compression method %d' % entry['method'])
        vf.seek(entry['start'])
        head = vf.read(30)
        if head[:4] != b'PK\x03\x04':
            raise RuntimeError('no local header where the directory points')
        n_name, n_extra = struct.unpack('<HH', head[26:30])
        self.at = entry['start'] + 30 + n_name + n_extra
        self.left = entry['csize']
        self.z = zlib.decompressobj(-15) if entry['method'] == 8 else None
        self.buf, self.crc, self.done, self.n_out = b'', 0, False, 0

    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False

    def close(self):
        pass

    def _more(self):
        import zlib
        if self.left <= 0:
            if self.z is not None and not self.done:
                self.buf += self.z.flush()
            self.done = True
            return
        self.vf.seek(self.at)
        raw = self.vf.read(min(self.left, 1 << 20))
        if not raw:
            raise RuntimeError('volume ends inside a member')
        self.at += len(raw)
        self.left -= len(raw)
        data = self.z.decompress(raw) if self.z is not None else raw
        if self.left <= 0 and self.z is not None:
            data += self.z.flush()
            self.done = True
        elif self.left <= 0:
            self.done = True
        self.crc = zlib.crc32(data, self.crc)
        self.n_out += len(data)
        self.buf += data
        if self.done and ((self.crc & 0xFFFFFFFF) != self.e['crc'] or self.n_out != self.e['usize']):
            raise zipfile.BadZipFile('Bad CRC-32 or length for member %r of the split archive' % self.e['name'])

    def read(self, n=-1):
        if n is None or n < 0:
            while not self.done:
                self._more()
            out, self.buf = self.buf, b''
            return out
        while len(self.buf) < n and not self.done:
            self._more()
        out, self.buf = self.buf[:n], self.buf[n:]
        return out


class SplitZipSource:
    """Members of a split zip read straight from its volumes: nothing is joined or rewritten on disk."""
    DIRECTORIES = {}                                          # read once; worker processes inherit it through fork

    def __init__(self, paths):
        self.paths = list(paths)
        self.listing = self._directory()
        self.entries = {e['name']: e for e in self.listing}   # a name that occurs twice gives its last copy, as zipfile does

    def _directory(self):
        key = '|'.join(self.paths)
        if key not in SplitZipSource.DIRECTORIES:
            vf = VolumeFile(self.paths)
            try:
                SplitZipSource.DIRECTORIES[key] = split_zip_directory(vf)
            finally:
                vf.close()
        return SplitZipSource.DIRECTORIES[key]

    def members(self):
        return [(e['name'], e['usize']) for e in self.listing if not e['name'].endswith('/')]

    def __getstate__(self):
        return {'paths': self.paths}                          # the directory can be large: it is not sent with every job

    def __setstate__(self, state):
        self.paths = state['paths']
        self.listing = self._directory()
        self.entries = {e['name']: e for e in self.listing}   # a name that occurs twice gives its last copy, as zipfile does

    def open(self):
        self._vf = VolumeFile(self.paths)
        return self

    def close(self):
        vf = self.__dict__.pop('_vf', None)
        if vf is not None:
            vf.close()

    def stream(self, name):
        return SplitMember(self._vf, self.entries[name])


def read_member(src, name, size):
    """Bytes of a member (gunzipped if needed). Very large members: first 2 MB and last 64 MB. Returns (bytes, partial flag)."""
    big = BIG_MEMBER_MB * 1024 * 1024
    with src.stream(name) as fh:
        if name.lower().endswith(('.gz', '.maegz')) and not name.lower().endswith('.tar.gz'):
            fh = gzip.GzipFile(fileobj=fh)
            size = None
        if size is not None and size <= big:
            return fh.read(), False
        head = fh.read(2 << 20)
        tail = collections.deque()
        kept = 0
        total = len(head)
        cut = False
        while True:
            chunk = fh.read(8 << 20)
            if not chunk:
                break
            total += len(chunk)
            tail.append(chunk)
            kept += len(chunk)
            if total > big:                                   # only now is the middle given up
                cut = True
                while kept - len(tail[0]) >= (64 << 20):
                    kept -= len(tail.popleft())
        if not cut:
            return head + b''.join(tail), False
        return head + b'\n' + b''.join(tail)[-(64 << 20):], True


def head_tail(blob, n_head=70, n_tail=25, width=200):
    lines = blob[:400000].decode('latin-1', 'replace').split('\n')
    if len(blob) <= 400000 and len(lines) <= n_head + n_tail:
        return '\n'.join(ln[:width] for ln in lines)
    tail_lines = blob[-60000:].decode('latin-1', 'replace').split('\n')[-n_tail:]
    return '\n'.join(ln[:width] for ln in lines[:n_head]) + '\n   [...]\n' + '\n'.join(ln[:width] for ln in tail_lines)


def mae_records(path, structures):
    """Records for the result files from the structures of one Maestro file: (one line per structure, geometries in the energy window)."""
    recs, geoms, ekey, emin = [], [], None, None
    for j, c in enumerate(structures):
        p = c['props']
        if j == 0:
            ekey = mae_energy_key(p)
        e = p.get(ekey) if ekey else None
        e = e if isinstance(e, (int, float)) else None
        keep_props = p if j == 0 else {a: b for a, b in p.items() if a.startswith(('r_mmod', 'i_mmod', 'b_mmod')) or a == 's_m_title'}
        rec = dict(path=path, ct=j, full=int(c['full']), n_atoms=c.get('n_atoms'), props=keep_props)
        if 'charge' in c:
            rec['charge'] = c['charge']
        if c.get('merged'):
            rec['merged'] = c['merged']
        if ekey:
            rec['e_key'] = ekey
        if 'xyz' in c:
            rec['pm'], rec['pmh'] = moments(c['xyz'], c.get('z'))
            if e is not None and (emin is None or e < emin):
                emin = e
                geoms = [g for g in geoms if g[0] is None or g[0] - emin <= MAE_WINDOW_KJ]
            if e is None or e - emin <= MAE_WINDOW_KJ:
                x = dict(path=path, ct=j, xyz=c['xyz'])
                if 'z' in c:
                    x['z'] = c['z']
                elif 'mmod_types' in c:
                    x['mmod_types'] = c['mmod_types']
                geoms.append((e, j, x))
                if len(geoms) > 4 * MAE_MAX_GEOMS:            # keep memory bounded on very long files
                    geoms = sorted(geoms, key=lambda g: (g[0] is None, g[0] or 0.0, g[1]))[:MAE_MAX_GEOMS]
        recs.append({a: b for a, b in rec.items() if b is not None})
    geoms = sorted(geoms, key=lambda g: (g[0] is None, g[0] or 0.0, g[1]))[:MAE_MAX_GEOMS]
    for r in recs:
        r['n_ct'] = len(recs)
    return recs, [x for _, _, x in sorted(geoms, key=lambda g: g[1])]


def work_chunk(args):
    """Parse a list of members of one source. Runs in a worker process."""
    src, tasks, prefix = args
    res = dict(g=[], gx=[], j=[], jraw=[], mae=[], maex=[], small=[], samples=[], errs=[], h=[], counts=collections.Counter())
    raw_n = collections.Counter()                             # outputs of each folder whose matrix text is kept (this chunk)
    try:
        src.open()
    except Exception as exc:
        res['errs'].append((prefix, 'cannot open source: %r' % (exc,)))
        return res
    for name, shown, size, kind, want_sample in tasks:
        path = prefix + shown
        try:
            if kind == 'qm':
                blob, partial = read_member(src, name, size)
                prog = sniff_program(blob)
                res['counts']['qm:' + prog] += 1
                if want_sample or (prog != 'gaussian' and res['counts']['qm:' + prog] == 1):
                    res['samples'].append((path, '%s output (%s)' % (prog, human(len(blob))), head_tail(blob)))
                if prog in ('nwchem', 'orca', 'jaguar', 'tinker', 'unknown') and len(blob) > 2000:
                    kept = harvest_lines(blob)
                    if len(kept) >= 2:
                        res['h'].append(dict(path=path, prog=prog, bytes=len(blob), lines=kept))
                if prog == 'gaussian':
                    folder = path.rpartition('/')[0]
                    core, extra = parse_gaussian(blob, True, raw_n[folder] < J_RAW_PER_FOLDER)
                    n_mat = 0
                    for r in core:
                        r['path'], r['bytes'] = path, len(blob)
                        if partial:
                            r['partial_read'] = 1
                        res['counts']['job:' + job_class(r.get('route'))] += 1
                        if 'iso' in r:
                            res['counts']['jobs with shieldings'] += 1
                        if r.get('norm'):
                            res['counts']['jobs ended normally'] += 1
                        js = r.pop('_j', None)
                        if js is not None and js['J']:        # coupling matrices: a record of their own
                            raw, bad = js.pop('raw'), js.pop('bad')
                            jrec = dict(path=path, job=r['job'], njobs=r['njobs'])
                            jrec.update({k2: r[k2] for k2 in ('route', 'scf', 'norm', 'nat', 'el') if k2 in r})
                            jrec.update(js)
                            if bad:
                                jrec['bad'] = bad
                            if partial:
                                jrec['partial_read'] = 1
                            res['j'].append(jrec)
                            n_mat += len(js['J'])
                            res['counts']['coupling: job steps with matrices'] += 1
                            res['counts']['coupling: J matrices kept'] += len(js['J'])
                            res['counts']['coupling: J matrices with an unexpected layout'] += len(bad)
                            for name, txt in raw.items():
                                res['jraw'].append(dict(path=path, job=r['job'], name=name, why='layout' if name in bad else 'sample', text=txt))
                    if n_mat:
                        res['counts']['coupling: outputs with matrices'] += 1
                        raw_n[folder] += 1
                    elif any('spinspin' in (r.get('route') or '').lower() for r in core):
                        res['counts']['coupling: spin-spin outputs WITHOUT a matrix'] += 1
                        if res['counts']['coupling: spin-spin outputs WITHOUT a matrix'] <= 2:
                            at = [i for i in (blob.find(b'spin-spin coupling'), blob.find(b'Fermi Contact')) if i >= 0]
                            res['samples'].append((path, 'spin-spin job WITHOUT a coupling matrix (%s)' % human(len(blob)),
                                                   head_tail(blob[min(at):min(at) + 30000], 60, 0) if at else head_tail(blob, 40, 80)))
                    for x in extra:
                        x['path'] = path
                    res['g'].extend(core)
                    res['gx'].extend(extra)
            elif kind == 'mae':
                gz = name.lower().endswith(('.gz', '.maegz'))
                if want_sample:
                    with src.stream(name) as fh:
                        head = (gzip.GzipFile(fileobj=fh) if gz else fh).read(60000)
                    res['samples'].append((path, 'Maestro file (%s)' % human(size), head_tail(head, 90, 0)))
                try:
                    with src.stream(name) as fh:
                        recs, geoms = mae_records(path, iter_mae(gzip.GzipFile(fileobj=fh) if gz else fh))
                except Exception as exc:                      # second, independent reader
                    first = '%s: %s' % (type(exc).__name__, str(exc)[:100])
                    blob, partial = read_member(src, name, size)
                    if partial:
                        raise
                    text = blob.decode('latin-1')
                    try:
                        recs, geoms = mae_records(path, iter_mae_lines(text))
                    except Exception as exc2:
                        if res['counts']['mae diagnostics (unread)'] < 2:
                            res['counts']['mae diagnostics (unread)'] += 1
                            res['samples'].append((path, 'Maestro file NOT READ (%s) | first reader: %s | second reader: %s: %s' % (
                                human(size), first, type(exc2).__name__, str(exc2)[:100]), mae_diag(text)))
                        raise RuntimeError('both Maestro readers failed | %s | %s: %s' % (first, type(exc2).__name__, str(exc2)[:100]))
                    for r in recs:
                        r['reader'] = 'lines'
                    res['counts']['mae files read by the second reader'] += 1
                    if res['counts']['mae diagnostics (second reader)'] < 1:
                        res['counts']['mae diagnostics (second reader)'] += 1
                        res['samples'].append((path, 'Maestro file read by the second reader (%s) | first reader: %s' % (human(size), first), mae_diag(text)))
                n_part = sum(1 for r in recs if not r['full'])
                if n_part and res['counts']['mae diagnostics (partial blocks)'] < 1:
                    res['counts']['mae diagnostics (partial blocks)'] += 1
                    with src.stream(name) as fh:
                        head = (gzip.GzipFile(fileobj=fh) if gz else fh).read(400000)
                    res['samples'].append((path, 'Maestro file with partial blocks (%s)' % human(size), mae_diag(head.decode('latin-1'))))
                res['mae'].extend(recs)
                res['maex'].extend(geoms)
                res['counts']['mae files'] += 1
                res['counts']['mae structures'] += len(recs)
                res['counts']['mae partial structures'] += n_part
                if n_part:
                    res['counts']['mae files with partial structures'] += 1
            elif kind == 'small':
                blob, _ = read_member(src, name, size)
                res['small'].append((path, blob))
                if want_sample:
                    res['samples'].append((path, 'small file (%s)' % human(len(blob)), head_tail(blob, 60, 10)))
            elif want_sample:
                with src.stream(name) as fh:
                    blob = fh.read(60000)
                res['samples'].append((path, 'first bytes of %s' % human(size), head_tail(blob, 40, 0)))
        except Exception as exc:
            res['errs'].append((path, '%s: %s' % (type(exc).__name__, str(exc)[:200])))
            res['counts']['members that failed to parse'] += 1
    src.close()
    return res


def join_split_zip(folder, stem, parts):
    """Turn stem.z01 ... stem.zip into one ordinary zip with the zip program. Returns the path, or None if that is not possible."""
    joined = os.path.join(folder, stem + '__joined.zip')
    last = os.path.join(folder, stem + '.zip')
    if not shutil.which('zip'):
        return None
    p = subprocess.run(['zip', '-q', '-s', '0', last, '--out', joined], cwd=folder, capture_output=True, text=True)
    if p.returncode == 0 and os.path.exists(joined) and zipfile.is_zipfile(joined):
        return joined
    log('   zip -s 0 failed:', (p.stderr or p.stdout)[:200])
    if os.path.exists(joined):
        os.remove(joined)
    return None


def split_volumes(parts):
    """Volumes in reading order: .z01, .z02, ..., then the .zip volume."""
    rest = [p for p in parts if not p.lower().endswith('.zip')]
    last = [p for p in parts if p.lower().endswith('.zip')]
    return sorted(rest, key=lambda p: int(re.search(r'\.z(\d+)$', p, re.I).group(1))) + last


def extract_tar(path, dest):
    os.makedirs(dest, exist_ok=True)
    root = os.path.realpath(dest)
    with tarfile.open(path) as tf:
        for m in tf:
            target = os.path.realpath(os.path.join(dest, m.name))
            if not m.isfile() or not (target == root or target.startswith(root + os.sep)):
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with tf.extractfile(m) as src_fh, open(target, 'wb') as dst_fh:
                shutil.copyfileobj(src_fh, dst_fh, 1 << 22)
    return dest

In [ ]:
# ---- 7. Result writers ----------------------------------------------------------------------------------------
class PartWriter:
    """Gzipped JSON lines, split into parts below the size of one result zip."""
    def __init__(self, folder, stem):
        self.folder, self.stem, self.k, self.n, self.fh, self.raw = folder, stem, 0, 0, None, None
        self.limit = int(PART_MB * 0.85 * 1024 * 1024)

    def _open(self):
        self.k += 1
        self.raw = open(os.path.join(self.folder, '%s_%03d.jsonl.gz' % (self.stem, self.k)), 'wb')
        self.fh = gzip.GzipFile(fileobj=self.raw, mode='wb', compresslevel=6, mtime=0)

    def write(self, obj):
        if self.fh is None:
            self._open()
        self.fh.write((json.dumps(obj, separators=(',', ':')) + '\n').encode('utf-8'))
        self.n += 1
        if self.n % 200 == 0 and self.raw.tell() > self.limit:
            self.close()

    def close(self):
        if self.fh is not None:
            self.fh.close()
            self.raw.close()
            self.fh = self.raw = None


class DepositOutput:
    def __init__(self, key, kinds=('qm', 'mae', 'small'), mode='parse', path_filter=None):
        self.key = key
        self.kinds = tuple(kinds)                             # member kinds that are read in this run
        self.mode = mode
        self.path_filter = re.compile(path_filter) if path_filter else None    # members outside it are not read
        self.j, self.jraw = PartWriter(CORE, 'jcoup_' + key), PartWriter(CORE, 'jraw_' + key)
        self.jraw_files, self.jraw_bytes = {}, 0
        self.sample_caps = collections.Counter()
        self.g, self.gx = PartWriter(CORE, 'gaussian_' + key), PartWriter(EXTRA, 'geom_' + key)
        self.mae, self.maex = PartWriter(CORE, 'mae_' + key), PartWriter(EXTRA, 'maegeom_' + key)
        self.harvest, self.harvest_bytes = PartWriter(CORE, 'otherqm_' + key), 0
        self.small_k, self.small_zip = 0, None
        self.small_seen, self.small_bytes, self.small_index = {}, 0, []
        self.paths = gzip.open(os.path.join(CORE if mode == 'jcoup' else EXTRA, 'paths_%s.tsv.gz' % key), 'wt', encoding='utf-8')
        self.dirs = collections.OrderedDict()
        self.samples, self.sample_kinds = [], set()
        self.counts = collections.Counter()
        self.errors = []
        self.census = collections.Counter()

    def note_member(self, path, size):
        self.paths.write('%s\t%d\n' % (path, size))
        d, _, base = path.rpartition('/')
        e = self.dirs.setdefault(d, dict(n=0, bytes=0, ext=collections.Counter(), ex=[]))
        e['n'] += 1
        e['bytes'] += size
        e['ext'][split_ext(path)[0] or '(none)'] += 1
        if len(e['ex']) < 3:
            e['ex'].append(base)

    def absorb(self, res):
        for r in res['g']:
            self.g.write(r)
            parts = r['path'].split('!/')[-1].split('/')
            self.census[('/'.join(parts[:-1])[-70:], job_class(r.get('route')), r.get('scf_m', '-'))] += 1
        if self.mode != 'jcoup':                              # the geometries of the coupling jobs were written in run 2
            for r in res['gx']:
                self.gx.write(r)
        for r in res.get('j', []):
            self.j.write(r)
        for r in res.get('jraw', []):
            if r['why'] == 'sample':                          # the first outputs of every folder, whatever the worker
                kept = self.jraw_files.setdefault(r['path'].rpartition('/')[0], [])
                if r['path'] not in kept:
                    if len(kept) >= J_RAW_PER_FOLDER:
                        continue
                    kept.append(r['path'])
            elif self.jraw_bytes + len(r['text']) > J_RAW_MB * 1024 * 1024:
                self.counts['coupling: matrix text left out (J_RAW_MB reached)'] += 1
                continue
            else:
                self.jraw_bytes += len(r['text'])
            self.jraw.write(r)
            self.counts['coupling: matrices with their text kept'] += 1
        for r in res.get('h', []):
            size = sum(len(x) for x in r['lines'])
            if self.harvest_bytes + size <= HARVEST_MB * 1024 * 1024 * 6:      # text shrinks about six times when packed
                self.harvest.write(r)
                self.harvest_bytes += size
                self.counts['other programs: outputs with lines kept'] += 1
            else:
                self.counts['other programs: outputs left out (HARVEST_MB reached)'] += 1
        for r in res['mae']:
            self.mae.write(r)
        for r in res['maex']:
            self.maex.write(r)
        for path, blob in res['small']:
            digest = hashlib.md5(blob).hexdigest()
            first = self.small_seen.get(digest)
            stored = 0
            if first is None and self.small_bytes + len(blob) <= SMALL_TOTAL_MB * 1024 * 1024:
                self.small_seen[digest] = path
                self.small_bytes += len(blob)
                if self.small_zip is None or self.small_zip.fp.tell() > PART_MB * 0.8 * 1024 * 1024:
                    if self.small_zip is not None:
                        self.small_zip.close()
                    self.small_k += 1
                    self.small_zip = zipfile.ZipFile(os.path.join(CORE, 'small_%s_%03d.zip' % (self.key, self.small_k)), 'w', zipfile.ZIP_DEFLATED)
                self.small_zip.writestr(path.replace('!/', '/'), blob)
                stored = 1
            self.small_index.append((path, len(blob), digest, stored, first or ''))
        for path, what, text in res['samples']:               # diagnostics and unparsed programs: a few examples are enough
            group = 'diag' if (what.startswith('Maestro file') and '(' in what and not what.startswith('Maestro file (')) \
                    or what.startswith('spin-spin job WITHOUT') else \
                    (what.split(' output (')[0] if ' output (' in what and not what.startswith('gaussian') else None)
            if group is not None:
                self.sample_caps[group] += 1
                if self.sample_caps[group] > (24 if group == 'diag' else 6):
                    continue
            self.samples.append((path, what, text))
        self.counts.update(res['counts'])
        self.errors.extend(res['errs'])

    def close(self):
        for w in (self.g, self.gx, self.j, self.jraw, self.mae, self.maex, self.harvest):
            w.close()
        if self.small_zip is not None:
            self.small_zip.close()
        self.paths.close()
        with gzip.open(os.path.join(CORE, 'small_index_%s.tsv.gz' % self.key), 'wt', encoding='utf-8') as fh:
            fh.write('path\tbytes\tmd5\tstored\tsame_content_as\n')
            for row in self.small_index:
                fh.write('%s\t%d\t%s\t%d\t%s\n' % row)
        with gzip.open(os.path.join(CORE, 'dirs_%s.tsv.gz' % self.key), 'wt', encoding='utf-8') as fh:
            fh.write('folder\tfiles\tbytes\textensions\texamples\n')
            for d, e in self.dirs.items():
                fh.write('%s\t%d\t%d\t%s\t%s\n' % (d, e['n'], e['bytes'], json.dumps(dict(e['ext'].most_common())), ' | '.join(e['ex'])))
        with gzip.open(os.path.join(CORE, 'samples_%s.txt.gz' % self.key), 'wt', encoding='utf-8') as fh:
            for path, what, text in self.samples:
                fh.write('=' * 110 + '\n%s\n%s\n' % (path, what) + '-' * 110 + '\n' + text + '\n\n')
        with open(os.path.join(CORE, 'errors_%s.txt' % self.key), 'w', encoding='utf-8') as fh:
            for path, msg in self.errors:
                fh.write('%s\t%s\n' % (path, msg))

In [ ]:
# ---- 8. Processing one source (archive or folder) -----------------------------------------------------------------
def sample_kind(path, kind):
    comps = path.split('/')
    parent = comps[-2].lower() if len(comps) > 1 else ''
    return (kind, split_ext(path)[0], parent if parent in ('nmr', 'opt', 'e', 'mm', 'opt_e') else '')


def process_source(src, prefix, out, depth=0):
    """Inventory every member, parse what is recognised, recurse into nested archives."""
    try:
        members = src.members()
    except Exception as exc:
        out.errors.append((prefix, 'cannot list: %r' % (exc,)))
        log('   cannot read', prefix, '|', repr(exc)[:160])
        return
    tasks, nested = [], []
    for name, size in members:
        shown = name.replace('\\', '/')                       # zips written on Windows may use backslashes
        kind = classify(shown, size)
        out.counts['members'] += 1
        if out.path_filter is not None and kind != 'archive' and not out.path_filter.search(prefix + shown):
            out.counts['members outside the filter (not read)'] += 1
            continue
        out.note_member(prefix + shown, size)
        out.counts['kind:' + kind] += 1
        sk = sample_kind(shown, kind)
        everything = len(out.kinds) > 1
        want = (everything or kind in out.kinds) and sk not in out.sample_kinds and len(out.sample_kinds) < 80
        if want:
            out.sample_kinds.add(sk)
        if kind == 'archive':
            nested.append((name, shown, size))
        elif kind in out.kinds or (want and everything and kind == 'other'):
            tasks.append((name, shown, size, kind, want))
    # chunks of roughly 48 MB or 300 members
    chunks, cur, cur_bytes = [], [], 0
    for t in tasks:
        cur.append(t)
        cur_bytes += t[2]
        if cur_bytes >= (48 << 20) or len(cur) >= 300:
            chunks.append(cur)
            cur, cur_bytes = [], 0
    if cur:
        chunks.append(cur)
    jobs = [(src, c, prefix) for c in chunks]
    done = False
    if N_WORKERS > 1 and len(jobs) > 1:
        try:
            ctx = multiprocessing.get_context('fork')
            with ctx.Pool(N_WORKERS) as pool:
                for res in pool.imap(work_chunk, jobs):
                    out.absorb(res)
            done = True
        except Exception as exc:
            log('   parallel run failed (%s); repeating this source on one core' % repr(exc)[:120])
            out.errors.append((prefix, 'parallel run failed: %r' % (exc,)))
    if not done:
        for job in jobs:
            out.absorb(work_chunk(job))
    for name, shown, size in nested:
        if depth >= 3:
            out.errors.append((prefix + shown, 'nested archive deeper than 3 levels: not opened'))
            continue
        if free_gb(TMP) * 1024 ** 3 < size * 6 + (1 << 30):
            out.errors.append((prefix + shown, 'not enough scratch space to open nested archive'))
            continue
        tmp = os.path.join(TMP, 'nested_%d_%s' % (depth, safe_name(shown)))
        try:
            with src.open().stream(name) as fh, open(tmp, 'wb') as dst:
                shutil.copyfileobj(fh, dst, 1 << 22)
            src.close()
            process_file(tmp, prefix + shown + '!/', out, depth + 1, display=shown.rsplit('/', 1)[-1])
        except Exception as exc:
            out.errors.append((prefix + shown, 'nested archive: %s' % str(exc)[:200]))
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)


def process_file(path, prefix, out, depth=0, display=None, relax=False):
    """Open one file from the repository (or a nested one) according to its type."""
    name = display or os.path.basename(path)
    kind = archive_kind(name)
    if kind == 'zip':
        if not zipfile.is_zipfile(path):
            raise RuntimeError('not a readable zip file')
        process_source(ZipSource(path, relax), prefix, out, depth)
    elif kind == 'tar':
        dest = os.path.join(TMP, 'untar_%d_%s' % (depth, safe_name(name)))
        try:
            process_source(DirSource(extract_tar(path, dest)), prefix, out, depth)
        finally:
            shutil.rmtree(dest, ignore_errors=True)
    elif kind == '7z':
        exe = shutil.which('7z') or shutil.which('7za') or shutil.which('7zr')
        if not exe:
            raise RuntimeError('7z archive and no 7z program on this machine')
        dest = os.path.join(TMP, 'un7z_%d_%s' % (depth, safe_name(name)))
        try:
            p = subprocess.run([exe, 'x', '-y', '-o' + dest, path], capture_output=True, text=True)
            if p.returncode != 0:
                raise RuntimeError('7z failed: ' + (p.stderr or p.stdout)[-200:])
            process_source(DirSource(dest), prefix, out, depth)
        finally:
            shutil.rmtree(dest, ignore_errors=True)
    else:                                                     # a loose file (for example References.txt)
        folder = os.path.join(TMP, 'loose_%d' % depth)
        shutil.rmtree(folder, ignore_errors=True)
        os.makedirs(folder)
        try:
            if os.path.getsize(path) <= (1 << 30):
                shutil.copy(path, os.path.join(folder, name))
                process_source(DirSource(folder), prefix, out, depth)
            else:
                out.note_member(prefix + name, os.path.getsize(path))
                out.counts['members'] += 1
                out.counts['kind:other'] += 1
        finally:
            shutil.rmtree(folder, ignore_errors=True)

In [ ]:
# ---- 9. Run: every deposit -------------------------------------------------------------------------------------
SUMMARY = dict(version=NOTEBOOK_VERSION, started=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), deposits=[])


def online():
    for url in ('https://pypi.org/simple/', 'https://www.kaggle.com/'):
        try:
            SESSION.get(url, timeout=10)
            return True
        except requests.RequestException:
            pass
    return False


if not LOCAL_SOURCES and not os.environ.get('CS_API_BASES') and not online():
    log('!' * 100)
    log('NO INTERNET CONNECTION. In the right panel open Settings and switch Internet on, then run again.')
    log('!' * 100)
SUMMARY['online'] = None if os.environ.get('CS_API_BASES') else online()
LOCAL = local_index()
if LOCAL:
    log('files found under the input folders:', len(LOCAL))


def run_deposit(dep):
    key = dep['key']
    info = dict(key=key, doi=dep.get('doi'), title=dep.get('title'), mode=dep.get('mode'), files=[])
    SUMMARY['deposits'].append(info)
    log('=' * 100)
    log('DEPOSIT', key, '|', dep.get('title'), '| doi', dep.get('doi'), '| mode', dep.get('mode'))
    mode = dep.get('mode')
    out = DepositOutput(key, kinds=('mae',) if mode == 'mae' else ('qm', 'small') if mode == 'jcoup' else ('qm', 'mae', 'small'),
                        mode=mode, path_filter=(dep.get('path_filter') or J_PATH) if mode == 'jcoup' else None)
    try:
        manual = LOCAL_SOURCES.get(key) or []
        if manual:
            info['listing'] = 'manual sources'
            for p in manual:
                frec = dict(name=os.path.basename(p.rstrip('/')), source=p)
                info['files'].append(frec)
                try:
                    if os.path.isdir(p):
                        process_source(DirSource(p), os.path.basename(p.rstrip('/')) + '/', out)
                    else:
                        process_file(p, os.path.basename(p) + '!/', out)
                    frec['status'] = 'parsed'
                except Exception as exc:
                    frec['status'] = 'failed: %s' % str(exc)[:200]
                    log('   FAILED', p, '|', str(exc)[:200])
            return
        files, how = list_record(dep)
        info['listing'] = how
        with open(os.path.join(CORE, 'files_%s.json' % key), 'w', encoding='utf-8') as fh:
            json.dump(dict(deposit=dep, listing=how, files=files), fh, indent=1)
        total = sum(f.get('size') or 0 for f in files)
        log('   listing: %s | %d files | %s in total' % (how, len(files), human(total)))
        for f in files:
            log('      %-52s %10s' % (f['name'][:52], human(f.get('size'))))
        info['n_files'], info['total_bytes'] = len(files), total
        known = {k[0]: k[2] for k in KNOWN_FILES.get(key, [])}
        for f in files:
            if f['name'] in known and f.get('size') not in (None, known[f['name']]):
                log('   note: %s is %s bytes in the API and %s on the record page' % (f['name'], f.get('size'), known[f['name']]))
        if not files:
            log('   NO FILE LIST. Manual route: download the files of doi', dep.get('doi'), 'in a browser, upload them as a Kaggle')
            log('   dataset, attach it to this notebook, set LOCAL_SOURCES = {"%s": ["/kaggle/input/<dataset>"]} and run again.' % key)
            return
        if dep.get('mode') not in ('parse', 'mae', 'jcoup'):
            for f in files:
                info['files'].append(dict(name=f['name'], size=f.get('size'), status='listed only'))
            return
        if total > MAX_DEPOSIT_GB * 1024 ** 3:
            log('   deposit is larger than MAX_DEPOSIT_GB = %s: listed only' % MAX_DEPOSIT_GB)
            for f in files:
                info['files'].append(dict(name=f['name'], size=f.get('size'), status='listed only (deposit too large)'))
            return
        # split zip sets: stem.z01, stem.z02, ..., stem.zip
        groups = collections.OrderedDict()
        for f in files:
            m = re.match(r'^(.*)\.z(\d{2,})$', f['name'], re.I)
            stem = m.group(1) if m else None
            groups.setdefault(('split', stem.lower()) if stem else ('single', f['name']), []).append(f)
        for (kind, gname), group in list(groups.items()):
            if kind == 'split':
                last = [g for g in files if g['name'].lower() == gname + '.zip']
                if last and ('single', last[0]['name']) in groups:
                    group.extend(groups.pop(('single', last[0]['name'])))
        for (kind, gname), group in groups.items():
            names = [f['name'] for f in group]
            frec = dict(name=' + '.join(names) if kind == 'split' else names[0], size=sum(f.get('size') or 0 for f in group))
            info['files'].append(frec)
            if ONLY_FILES and not any(n in ONLY_FILES for n in names):
                frec['status'] = 'skipped (not in ONLY_FILES)'
                continue
            if dep.get('only_names') and not any(n in dep['only_names'] for n in names):
                frec['status'] = 'left out (done in run 1)'
                continue
            if any(n in (dep.get('skip_names') or []) for n in names):
                frec['status'] = 'skipped (skip_names)'
                continue
            if hours() > DEADLINE_HOURS:
                frec['status'] = 'not started: time limit reached'
                continue
            if frec['size'] > MAX_FILE_GB * 1024 ** 3:
                frec['status'] = 'skipped: larger than MAX_FILE_GB'
                continue
            need = frec['size'] * (2.2 if kind == 'split' else 1.1) + (2 << 30)
            if free_gb(DL) * 1024 ** 3 < need:
                frec['status'] = 'skipped: not enough scratch space (%.1f GB free)' % free_gb(DL)
                log('   SKIP', frec['name'], '|', frec['status'])
                continue
            t1 = time.time()
            folder = os.path.join(DL, key, safe_name(gname))
            shutil.rmtree(folder, ignore_errors=True)
            os.makedirs(folder)
            try:
                paths, notes = [], []
                for f in group:
                    dest = os.path.join(folder, f['name'])
                    if f['name'] in LOCAL and (not f.get('size') or os.path.getsize(LOCAL[f['name']]) == f['size']):
                        dest, note = LOCAL[f['name']], 'taken from the input folder'
                    else:
                        ok, note = download(f, dest)
                        if not ok:
                            raise RuntimeError('download failed for %s: %s' % (f['name'], note))
                    paths.append(dest)
                    notes.append(note)
                frec['download'] = '; '.join(sorted(set(notes)))
                t2 = time.time()
                before = collections.Counter(out.counts)
                if kind == 'split':
                    for p in paths:                           # volumes taken from the input folder are copied next to the others
                        if os.path.dirname(p) != folder:
                            shutil.copy(p, os.path.join(folder, os.path.basename(p)))
                    last = [f['name'] for f in group if f['name'].lower().endswith('.zip')]
                    if not last:
                        raise RuntimeError('split zip set without its final .zip volume')
                    stem = last[0][:-4]
                    vols = split_volumes([os.path.join(folder, n) for n in names])
                    try:                                      # read the volumes directly: no second copy on disk
                        split_src = SplitZipSource(vols)
                        frec['split_reader'] = 'volumes read directly (%d entries)' % len(split_src.listing)
                        process_source(split_src, stem + '.zip!/', out)
                    except Exception as exc:
                        log('   direct reading of the volumes failed (%s); joining them with the zip program' % str(exc)[:160])
                        target = join_split_zip(folder, stem, vols)
                        if target is None:
                            raise RuntimeError('split zip set could not be read: %s' % str(exc)[:200])
                        frec['split_reader'] = 'joined with the zip program'
                        process_file(target, stem + '.zip!/', out, display=stem + '.zip', relax=True)
                else:
                    process_file(paths[0], names[0] + '!/' if archive_kind(names[0]) else '', out, display=names[0])
                delta = out.counts - before
                frec['status'] = 'parsed'
                frec['members'] = delta.get('members', 0)
                frec['gaussian_jobs'] = sum(v for k2, v in delta.items() if k2.startswith('job:'))
                frec['mae_structures'] = delta.get('mae structures', 0)
                frec['j_matrices'] = delta.get('coupling: J matrices kept', 0)
                log('   ok  %-40s %9s | download %4.0fs (%s) | parse %4.0fs | members %6d | Gaussian jobs %6d | Maestro structures %6d | J matrices %6d'
                    % (frec['name'][:40], human(frec['size']), t2 - t1, frec['download'], time.time() - t2,
                       frec['members'], frec['gaussian_jobs'], frec['mae_structures'], frec['j_matrices']))
            except Exception as exc:
                frec['status'] = 'failed: %s' % str(exc)[:300]
                out.errors.append((frec['name'], frec['status']))
                log('   FAILED', frec['name'], '|', str(exc)[:300])
                log('   ' + traceback.format_exc().strip().splitlines()[-1][:300])
            finally:
                shutil.rmtree(folder, ignore_errors=True)
    finally:
        out.close()
        info['counts'] = dict(sorted(out.counts.items()))
        info['n_errors'] = len(out.errors)
        info['census'] = [[d, c, m, n] for (d, c, m), n in sorted(out.census.items())][:4000]
        if out.counts:
            log('   totals:', ' | '.join('%s %d' % kv for kv in sorted(out.counts.items())))


for dep in DEPOSITS:
    if dep.get('mode') == 'off':
        continue
    try:
        run_deposit(dep)
    except Exception as exc:
        log('DEPOSIT', dep.get('key'), 'stopped by an error:', repr(exc)[:300])
        log(traceback.format_exc()[-1500:])
SUMMARY['hours'] = round(hours(), 2)
with open(os.path.join(CORE, 'summary.json'), 'w', encoding='utf-8') as fh:
    json.dump(SUMMARY, fh, indent=1)
log('all deposits done after %.2f h' % hours())

In [ ]:
# ---- 10. Package the results into zips of at most PART_MB --------------------------------------------------------
def pack(folder, stem):
    """Greedy packing of the files of a folder into stem_NN.zip in WORK. Large files get their own zip."""
    limit = int(PART_MB * 1024 * 1024)
    files = sorted((os.path.getsize(os.path.join(folder, f)), f) for f in os.listdir(folder))
    files = [(s, f) for s, f in files if s > 0 or f.endswith('.txt')]
    bins = []
    for size, f in sorted(files, key=lambda t: (-t[0], t[1])):
        for b in bins:
            if b['bytes'] + size <= limit:
                b['files'].append(f)
                b['bytes'] += size
                break
        else:
            bins.append(dict(files=[f], bytes=size))
    made = []
    for k, b in enumerate(bins, 1):
        target = os.path.join(WORK, '%s_%02d.zip' % (stem, k))
        with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for f in sorted(b['files']):
                stored = f.endswith(('.gz', '.zip'))
                zf.write(os.path.join(folder, f), f, zipfile.ZIP_STORED if stored else zipfile.ZIP_DEFLATED)
        made.append((os.path.basename(target), os.path.getsize(target), sorted(b['files'])))
    return made


for old in os.listdir(WORK):
    if re.match(r'^cs1[12]_(core|extra)_\d+\.zip$', old):
        os.remove(os.path.join(WORK, old))
log('packaging')
with open(os.path.join(CORE, 'MANIFEST.txt'), 'w', encoding='utf-8') as fh:
    fh.write('ConformalStereo Cambridge deposits (run 3), notebook %s, written %s\n\n' % (NOTEBOOK_VERSION, time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())))
    for label, folder in (('core', CORE), ('extra', EXTRA)):
        fh.write('[%s]\n' % label)
        for f in sorted(os.listdir(folder)):
            if f != 'MANIFEST.txt':
                fh.write('  %-46s %12d bytes\n' % (f, os.path.getsize(os.path.join(folder, f))))
core_zips = pack(CORE, 'cs12_core')
extra_zips = pack(EXTRA, 'cs12_extra')

print()
print('=' * 100)
print('RESULT FILES in', WORK)
for name, size, _ in core_zips:
    print('   SEND THIS   %-22s %10s' % (name, human(size)))
for name, size, _ in extra_zips:
    print('   SEND THIS   %-22s %10s   (conformer geometries)' % (name, human(size)))
print()
for info in SUMMARY['deposits']:
    c = info.get('counts') or {}
    st = collections.Counter((f.get('status') or '?').split(':')[0] for f in info.get('files', []))
    print('%-9s %-28s files: %s' % (info['key'], '(' + str(info.get('listing')) + ')', ', '.join('%d %s' % (n, s) for s, n in st.items()) or 'none'))
    if c and info.get('mode') == 'jcoup':
        print('          coupling constants: %d outputs with matrices (%d job steps) | J matrices kept: %d | unexpected layout: %d' % (
            c.get('coupling: outputs with matrices', 0), c.get('coupling: job steps with matrices', 0),
            c.get('coupling: J matrices kept', 0), c.get('coupling: J matrices with an unexpected layout', 0)))
        print('          spin-spin outputs without a matrix: %d | matrices with their text kept: %d | members read: %d of %d | parse errors: %d' % (
            c.get('coupling: spin-spin outputs WITHOUT a matrix', 0), c.get('coupling: matrices with their text kept', 0),
            c.get('members', 0) - c.get('members outside the filter (not read)', 0) - c.get('kind:archive', 0), c.get('members', 0),
            info.get('n_errors', 0)))
    elif c and info.get('mode') == 'mae':
        print('          Maestro files: %d with %d structures | members listed: %d | parse errors: %d' % (
            c.get('mae files', 0), c.get('mae structures', 0), c.get('members', 0), info.get('n_errors', 0)))
        print('          files that hold partial structures: %d (%d structures) | files read by the second reader: %d' % (
            c.get('mae files with partial structures', 0), c.get('mae partial structures', 0), c.get('mae files read by the second reader', 0)))
    elif c:
        jobs = {k[4:]: v for k, v in c.items() if k.startswith('job:')}
        print('          Gaussian job steps: %d %s | with shieldings: %d | ended normally: %d' % (
            sum(jobs.values()), jobs, c.get('jobs with shieldings', 0), c.get('jobs ended normally', 0)))
        print('          Maestro files: %d with %d structures | small files copied: %d | members in all: %d | parse errors: %d' % (
            c.get('mae files', 0), c.get('mae structures', 0), c.get('kind:small', 0), c.get('members', 0), info.get('n_errors', 0)))
        if c.get('mae partial structures') or c.get('mae files read by the second reader'):
            print('          Maestro: %d files hold partial structures (%d structures) | files read by the second reader: %d' % (
                c.get('mae files with partial structures', 0), c.get('mae partial structures', 0), c.get('mae files read by the second reader', 0)))
        other = {k[3:]: v for k, v in c.items() if k.startswith('qm:') and k != 'qm:gaussian'}
        if other:
            print('          other .out/.log files (not parsed):', other)
            if c.get('other programs: outputs with lines kept'):
                print('          ... energy and shielding lines kept for %d of them (otherqm_%s files)' % (c['other programs: outputs with lines kept'], info['key']))
    for f in info.get('files', []):
        if str(f.get('status', '')).startswith(('failed', 'skipped', 'not started')):
            print('          !! %-40s %s' % (f['name'][:40], f['status'][:120]))
warn = []
for info in SUMMARY['deposits']:
    c = info.get('counts') or {}
    if info.get('mode') == 'mae' and not c.get('mae structures'):
        warn.append('%s: no Maestro structure was read' % info['key'])
    if info.get('mode') == 'parse' and not any(k.startswith('job:') for k in c):
        warn.append('%s: no Gaussian job was parsed' % info['key'])
    if info.get('mode') == 'jcoup' and not c.get('coupling: J matrices kept'):
        warn.append('%s: no coupling matrix was read' % info['key'])
    if c.get('coupling: J matrices with an unexpected layout'):
        warn.append('%s: %d coupling matrices have an unexpected layout (their text is in jraw_%s_NNN.jsonl.gz)' % (
            info['key'], c['coupling: J matrices with an unexpected layout'], info['key']))
    if c.get('coupling: spin-spin outputs WITHOUT a matrix'):
        warn.append('%s: %d spin-spin outputs print no coupling matrix (two examples are in samples_%s.txt.gz)' % (
            info['key'], c['coupling: spin-spin outputs WITHOUT a matrix'], info['key']))
    if str(info.get('listing', '')).startswith(('fallback', 'none')):
        warn.append('%s: the repository API did not answer; file list is %s' % (info['key'], info.get('listing')))
print()
print('WARNINGS:' if warn else 'No warnings.')
for w in warn:
    print('   -', w)
print('Finished in %.2f h.' % hours())